# Costruzione del panel

Costruisce il panel azienda-anno dai CSV di PitchBook: una riga per ogni anno di
vita di ogni azienda, dall'anno di fondazione all'ultimo anno in cui esiste un
dato su di lei. È l'input della costruzione dei dataset e degli esperimenti, in
`notebook.ipynb`.

Il risultato ha **52 colonne**: le 50 dello schema di `data/raw/example_panel.csv`
— con `TotalRaised` al posto di `TotalRaised_Est` — più `UndisclosedAmountShare`
e `N_Similar`. Gli output di ogni fase vanno in `data/interim/`; il panel finale
è `panel.parquet` e `panel.csv.gz`.

## Le fasi

| fase | cosa costruisce |
|---|---|
| **1** | lo scheletro: una riga per azienda-anno |
| **2** | la tabella persona-azienda, con la finestra di presenza di ognuno |
| **3** | le colonne di team, anno per anno |
| **4** | i round di finanziamento e gli investitori |
| **5** | gli stadi di crescita, le cumulate e il CEO |
| **6** | il raggruppamento degli stadi e il troncamento all'uscita |
| **7** | i concorrenti |

## Cosa vale «mancante»

Ogni CSV si legge come testo, senza inferenza di tipo: un identificativo che
sembra numerico, convertito in numero, romperebbe i join in silenzio. I
segnaposto testuali (`""`, `"NA"`, `"N/A"`, `"NULL"`, `"NaN"`) diventano null
subito dopo la lettura, altrimenti in una colonna categoriale diventerebbero una
categoria a sé.

Un secondo insieme di token si applica **dopo** le aggregazioni e comprende le
infinità: `max()` su un gruppo tutto vuoto restituisce `-Inf` e `mean()`
restituisce `NaN`, che sono artefatti del calcolo e non valori letti dai file.

## Gli interruttori della temporizzazione

Tre interruttori nella cella di import scelgono, per un gruppo di attributi
ciascuno, fra **il valore dell'anno della riga** e **la fotografia scattata alla
data di estrazione**:

- `TEMPORIZZA_PERSONE` — esperienza e istruzione di team e CEO;
- `TEMPORIZZA_INVESTITORI` — dimensione e attività degli investitori;
- `TEMPORIZZA_COMPETITOR` — concorrenti e similarità.

Servono a produrre i due panel da confrontare, con e senza look-ahead, a parità
di tutto il resto: stessa pipeline, stesse feature, cambia solo cosa si sapeva
all'epoca. Ogni interruttore governa il suo gruppo e nient'altro.

`TEMPORIZZA_INVESTITORI` parte spento: la versione anno per anno vede solo i deal
delle aziende dell'estrazione, quindi sottostima i fondi grandi.

In [1]:
# Import, percorsi e interruttori della temporizzazione
import gc
import sys
from pathlib import Path

import polars as pl
import yaml

ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.panel.config import PanelConfig
from src.panel.expansions import active_pairs, expand_team
from src.panel.io import COMPANY_DATE_COLUMNS, read_raw, to_num
from src.panel.expressions import (
    MISSING_TOKENS,
    MISSING_TOKENS_WITH_INF,
    nullify,
    cumulative_any,
    next_different,
    parse_date,
    first_match,
    cum_sum_null,
    if_else_null,
    seq_inclusive,
    last_non_null,
    weighted_cumulative,
)

cfg = PanelConfig()

pl.Config.set_tbl_cols(12)
pl.Config.set_fmt_str_lengths(40)

with open(ROOT / "config" / "config.yaml") as f:
    CONFIG = yaml.safe_load(f)
ANNO_MIN_FONDAZIONE = int(CONFIG["first_year"])

print(f"anno di fondazione minimo: {ANNO_MIN_FONDAZIONE} (incluso), da config.yaml")
print("output di stadio         :", cfg.interim_dir)

TEMPORIZZA_PERSONE = True
TEMPORIZZA_INVESTITORI = False
TEMPORIZZA_COMPETITOR = True
print("temporizzazione          : persone", TEMPORIZZA_PERSONE,
      "\u00b7 investitori", TEMPORIZZA_INVESTITORI,
      "\u00b7 competitor", TEMPORIZZA_COMPETITOR)

anno di fondazione minimo: 2000 (incluso), da config.yaml
output di stadio         : data/interim
temporizzazione          : persone True · investitori False · competitor True


---
## Fase 1 — lo scheletro del panel

Da una riga per azienda a **una riga per azienda-anno**: è qui che nasce la forma
del panel.

Di `Company.csv` servono undici colonne. Quattro non producono nessuna colonna
finale: insieme a `FiscalPeriod`, che porta una data di bilancio in forma
testuale, servono solo a calcolare `MaxYear`.

**`MaxYear` è l'ultimo anno in cui PitchBook ha un dato su quell'azienda**, cioè
il più recente fra sei date. È lì che finisce il panel di quell'azienda: dopo di
quello non si sa più niente, quindi non si potrebbe nemmeno dire come è andata.
Un'azienda senza anno di fondazione o senza `MaxYear` non entra nel panel.

Il limite superiore dello scheletro è il maggiore fra `MaxYear` e l'anno di
fondazione, perché qualche scheda ha date anteriori alla fondazione — tipicamente
ri-registrazioni — e senza quel limite genererebbe anni precedenti alla nascita
dell'azienda.

Lo **stato di proprietà** (`Privately Held`, `Acquired/Merged`, `Out of
Business`) è una riga per azienda, cioè lo stato attuale, e si aggancia al solo
anno della propria data: su tutti gli altri anni resta vuoto. Servirà alla
cascata degli stadi.

L'ultima cella tiene solo le aziende fondate dall'anno minimo in poi. La soglia
sta in `config/config.yaml` perché è una scelta di campione, e lo stesso valore è
usato in tutti e tre i punti che filtrano le aziende: finché la sorgente è una,
le fasi non possono divergere.

In [2]:
# 1.1 · le colonne di Company.csv
COLONNE_COMPANY = [
    "CompanyID",
    "YearFounded",
    "HQCountry",
    "PrimaryIndustrySector",
    "OwnershipStatus",
    "OwnershipStatusDate",
    "CompanyFinancingStatusDate",
    "BusinessStatusDate",
    "FirstFinancingDate",
    "LastKnownValuationDate",
    "FiscalPeriod",
]

azienda = read_raw(cfg, "Company", COLONNE_COMPANY)

azienda = nullify(azienda, MISSING_TOKENS)

print(f"{azienda.height:,} aziende x {azienda.width} colonne")

134,355 aziende x 11 colonne


In [3]:
# 1.2 · le date, e la data di bilancio
azienda = azienda.with_columns(
    *[parse_date(pl.col(c)).alias(c) for c in COMPANY_DATE_COLUMNS],
    pl.col("YearFounded").cast(pl.Int64, strict=False),
)

trimestre = pl.col("FiscalPeriod").str.extract(r"TTM (\d)Q\d{4}", 1).cast(pl.Int64, strict=False)
anno_fiscale = pl.col("FiscalPeriod").str.slice(-4).cast(pl.Int64, strict=False)
# "TTM 2Q2019" -> 30 giugno 2019: il trimestre diventa il mese di chiusura.
azienda = azienda.with_columns(pl.date(anno_fiscale, trimestre * 3, 30).alias("FiscalDate"))

print(f"FiscalDate valorizzata su {azienda['FiscalDate'].is_not_null().sum():,} aziende")

FiscalDate valorizzata su 90,712 aziende


In [4]:
# 1.3 · l'ultimo anno di dati, e lo scheletro del panel
DATE_MAXYEAR = [*COMPANY_DATE_COLUMNS, "FiscalDate"]

vita = (
    azienda
    .with_columns(pl.max_horizontal([pl.col(c).dt.year() for c in DATE_MAXYEAR]).alias("MaxYear"))
    .filter(pl.col("YearFounded").is_not_null() & pl.col("MaxYear").is_not_null())
    .select("CompanyID", "YearFounded", "MaxYear")
)
print(f"aziende con anno di fondazione e MaxYear: {vita.height:,} su {azienda.height:,}")

limite = pl.max_horizontal("MaxYear", "YearFounded")

scheletro = (
    vita
    .with_columns(seq_inclusive(pl.col("YearFounded"), limite).alias("Year_Delta"))
    .explode("Year_Delta", empty_as_null=True)
    .with_columns((pl.col("Year_Delta") - pl.col("YearFounded")).alias("Delta"))
    .select("CompanyID", "YearFounded", "Year_Delta", "Delta")
)
print(f"scheletro: {scheletro.height:,} righe azienda-anno")
print(f"righe con Delta < 0 (deve essere 0): {scheletro.filter(pl.col('Delta') < 0).height}")

aziende con anno di fondazione e MaxYear: 126,817 su 134,355


scheletro: 1,309,093 righe azienda-anno
righe con Delta < 0 (deve essere 0): 0


In [5]:
# 1.4 · lo stato di proprieta', nell'anno della sua data
anno_proprieta = (
    azienda
    .select(
        "CompanyID",
        pl.col("OwnershipStatusDate").dt.year().alias("_anno_own"),
        "OwnershipStatus",
    )
    .drop_nulls("_anno_own")
)

scheletro = scheletro.join(
    anno_proprieta,
    left_on=["CompanyID", "Year_Delta"],
    right_on=["CompanyID", "_anno_own"],
    how="left",
)
print(f"righe con OwnershipStatus valorizzato: {scheletro['OwnershipStatus'].is_not_null().sum():,}"
      f" su {scheletro.height:,}")

righe con OwnershipStatus valorizzato: 126,416 su 1,309,093


In [6]:
# 1.5 · il filtro sull'anno di fondazione, e la scrittura
db_master_1 = azienda.select(
    "CompanyID", "YearFounded", "HQCountry", "PrimaryIndustrySector",
    "OwnershipStatus", "OwnershipStatusDate",
).filter(pl.col("YearFounded") >= ANNO_MIN_FONDAZIONE)

scheletro = scheletro.filter(pl.col("YearFounded") >= ANNO_MIN_FONDAZIONE)

db_master_1.write_parquet(cfg.interim("db_master_1.parquet"))
scheletro.write_parquet(cfg.interim("scheletro.parquet"))

print(f"db_master_1: {db_master_1.height:,} righe x {db_master_1.width} colonne  (attese 116.920)")
print(f"scheletro  : {scheletro.height:,} righe x {scheletro.width} colonne")

del azienda, vita, anno_proprieta, db_master_1
gc.collect()

db_master_1: 116,920 righe x 6 colonne  (attese 116.920)
scheletro  : 907,934 righe x 5 colonne


0

---
## Fase 2 — la tabella persona-azienda

Una riga per **coppia (azienda, persona)**, con gli attributi di quella persona e
la finestra di anni in cui era presente in quell'azienda.

Entrano solo le aziende dello scheletro. La tabella dei board team ha una riga
per **incarico**, quindi la stessa persona compare più volte se ha avuto più
ruoli nella stessa azienda, o se lo stesso ruolo è registrato due volte. Le
variabili di team sono per persona e non per ruolo, quindi le righe si fondono in
una sola, con due regole diverse:

- **stesso incarico** (stessa coppia e stesso titolo): è la stessa informazione
  registrata due volte, quindi si tiene la versione più affidabile. «In carica»
  vince su una data di uscita, una data vince su un valore mancante, e a parità
  di ultimo aggiornamento si tiene l'intervallo più ampio;
- **ruoli diversi** della stessa coppia: conta in quali anni la persona c'era,
  quindi si prende l'**unione** dei periodi. I titoli si concatenano, così chi è
  fondatore in uno dei ruoli resta fondatore. Se fra due ruoli c'è un buco, la
  persona risulta presente anche negli anni scoperti: sono pochissimi casi, e
  tenere i ruoli separati fino all'espansione costerebbe più di quanto valga.

Nella stessa cella si mette da parte la tabella dei **ruoli da CEO** con le loro
finestre, perché più avanti le date della permanenza diventano età dell'azienda e
il titolo non basterebbe più. Attenzione a cosa significano quelle date: dicono
quando la persona è entrata in azienda, non quando è diventata CEO.

`db3` esce con dieci colonne: la coppia, il genere, i titoli ricavati dal nome,
se è fondatore, l'anno di fondazione e la finestra di presenza in età
dell'azienda.

In [7]:
# 2.1 · una riga per coppia azienda-persona, e i ruoli da CEO
COLONNE_BOARD = [
    "CompanyID", "PersonID",
    "PersonName",
    "FullTitle",
    "IsCurrent",
    "StartDate", "EndDate",
]
COPPIA = ["CompanyID", "PersonID"]

board = nullify(read_raw(cfg, "CompanyBoardTeamRelation", [*COLONNE_BOARD, "LastUpdated"]), MISSING_TOKENS)

vita_azienda = (
    pl.read_parquet(cfg.interim("scheletro.parquet"))
    .group_by("CompanyID")
    .agg(pl.col("YearFounded").first(), pl.col("Year_Delta").max().alias("UltimoAnno"))
)
print(f"righe grezze: {board.height:,}", end="   ")
board = board.join(vita_azienda.select("CompanyID"), on="CompanyID", how="semi")
print(f"nelle aziende dello scheletro: {board.height:,}   coppie distinte: {board.select(COPPIA).n_unique():,}")

STESSO_INCARICO = ["CompanyID", "PersonID", "PersonName", "FullTitle"]
aggiornata = parse_date(pl.col("LastUpdated"))
in_carica = (pl.col("IsCurrent") == "Yes").any()
multiple = pl.len() > 1

def data_scelta(colonna: str, spareggio: str) -> pl.Expr:
    """La data della riga aggiornata piu' di recente fra quelle che ne hanno una;
    a parita' di LastUpdated, la min (StartDate) o la max (EndDate)."""
    data = parse_date(pl.col(colonna))
    ha_data = data.is_not_null()
    candidate = data.filter(ha_data & (aggiornata == aggiornata.filter(ha_data).max()))
    scelta = candidate.min() if spareggio == "min" else candidate.max()
    return scelta.dt.strftime("%m/%d/%Y")

prima = board.height
board = (
    board.with_row_index("_riga")
    .group_by(STESSO_INCARICO)
    .agg(
        pl.col("_riga").min(),
        pl.when(multiple & in_carica).then(pl.lit("Yes"))
        .otherwise(pl.col("IsCurrent").drop_nulls().first())
        .alias("IsCurrent"),
        pl.when(multiple).then(data_scelta("StartDate", "min"))
        .otherwise(pl.col("StartDate").first())
        .alias("StartDate"),
        pl.when(multiple & in_carica).then(pl.lit(None, dtype=pl.String))
        .when(multiple).then(data_scelta("EndDate", "max"))
        .otherwise(pl.col("EndDate").first())
        .alias("EndDate"),
    )
    .sort("_riga")
    .select(COLONNE_BOARD)
)
print(f"righe fuse perche' stesso incarico: {prima - board.height:,}")

inizio, fine = parse_date(pl.col("StartDate")), parse_date(pl.col("EndDate"))
prima = board.height
board = (
    board.with_row_index("_riga")
    .group_by(COPPIA)
    .agg(
        pl.col("_riga").min(),
        pl.col("PersonName").first(),
        pl.when(pl.col("FullTitle").is_not_null().any())
        .then(pl.col("FullTitle").drop_nulls().unique(maintain_order=True).str.join(", "))
        .alias("FullTitle"),
        pl.when(multiple & in_carica).then(pl.lit("Yes"))
        .otherwise(pl.col("IsCurrent").drop_nulls().first())
        .alias("IsCurrent"),
        pl.when(multiple & inizio.is_null().any()).then(pl.lit(None, dtype=pl.String))
        .when(multiple).then(inizio.min().dt.strftime("%m/%d/%Y"))
        .otherwise(pl.col("StartDate").first())
        .alias("StartDate"),
        pl.when(multiple & (in_carica | fine.is_null().any())).then(pl.lit(None, dtype=pl.String))
        .when(multiple).then(fine.max().dt.strftime("%m/%d/%Y"))
        .otherwise(pl.col("EndDate").first())
        .alias("EndDate"),
    )
    .sort("_riga")
    .select(COLONNE_BOARD)
)
print(f"righe fuse perche' ruoli diversi della stessa coppia: {prima - board.height:,}")

assert board.height == board.select(COPPIA).n_unique(), "restano coppie su piu' righe"
print(f"dopo la deduplica: {board.height:,} righe, una per coppia")

_titolo = pl.col("FullTitle").fill_null("")
_senza_assistenti = _titolo.str.replace_all(r"(?i)founders?'?s?'? associate", "")
_sv = parse_date(pl.col("StartDate")).dt.year()
_ev = parse_date(pl.col("EndDate")).dt.year()

ruoli_ceo = (
    board.filter(_titolo.str.contains(r"(?i)\bceo\b|chief executive"))
    .join(vita_azienda, on="CompanyID", how="inner")
    .with_columns(
        _sv.alias("sv"),
        _senza_assistenti.str.contains(r"(?i)founde|founding").alias("is_founder"),
    )
    .with_columns(
        pl.max_horizontal(pl.coalesce("sv", "YearFounded"), pl.col("YearFounded")).alias("da"),
        pl.min_horizontal(pl.coalesce(_ev, pl.col("UltimoAnno")), pl.col("UltimoAnno")).alias("a"),
    )
    .filter(pl.col("da") <= pl.col("a"))
    .select("CompanyID", "PersonID", "da", "a", "sv", "is_founder")
)
ruoli_ceo.write_parquet(cfg.interim("ruoli_ceo.parquet"))
print(f"ruoli da CEO nel board: {ruoli_ceo.height:,}  "
      f"(founder+CEO: {ruoli_ceo['is_founder'].sum():,}, con data vera: {ruoli_ceo['sv'].is_not_null().sum():,})")

righe grezze: 535,568   

nelle aziende dello scheletro: 466,312   coppie distinte: 465,861


righe fuse perche' stesso incarico: 158


righe fuse perche' ruoli diversi della stessa coppia: 293
dopo la deduplica: 465,861 righe, una per coppia


ruoli da CEO nel board: 94,464  (founder+CEO: 67,410, con data vera: 79,873)


In [8]:
# 2.2 · le date della permanenza
db3 = (
    board
    .with_columns(parse_date(pl.col(c)).alias(c) for c in ("StartDate", "EndDate"))
    .sort(["CompanyID", "StartDate"], nulls_last=True)
)
del board
gc.collect()
print(f"StartDate valorizzate: {db3['StartDate'].is_not_null().sum():,} su {db3.height:,}")

StartDate valorizzate: 324,481 su 465,861


In [9]:
# 2.3 · il genere
persona = nullify(read_raw(cfg, "Person", ["PersonID", "Gender"]), MISSING_TOKENS)
db3 = db3.join(persona, on="PersonID", how="left")
del persona
gc.collect()
print(f"righe senza genere: {db3['Gender'].is_null().sum():,} su {db3.height:,}")

righe senza genere: 2,683 su 465,861


### L'esperienza, anno per anno

`Person.csv` espone otto contatori di ruoli, ma sono totali alla data di
estrazione: non dicono quanti ruoli quella persona avesse nel 2012. Ognuno si
ricostruisce quindi da una tabella di dettaglio con una riga per ruolo:
posizioni, seggi nei consigli, incarichi da advisor, deal e fondi affiliati.

**Ogni ruolo diventa un evento con un anno**, e l'esperienza all'anno Y è il
numero di ruoli iniziati entro Y, finiti o no: la data di fine non serve. L'anno
di un ruolo si sceglie in cascata:

1. la data dichiarata;
2. se manca, l'anno di fondazione dell'entità in cui si svolge: non è la data
   vera, ma un ruolo non può iniziare prima che l'entità esista;
3. se manca anche quello, il primo anno noto della persona, cioè il più antico
   fra quelli risolti ai due punti precedenti sui suoi altri ruoli. Le entità che
   arrivano qui sono aziende fuori dall'estrazione o entità-persona come gli
   angel, e non hanno una fondazione a cui ancorarsi;
4. se la persona non ha nemmeno un altro ruolo datato, l'anno 0, cioè «conta in
   ogni anno».

Per i fondi una data dichiarata non esiste: l'affiliazione non può precedere né
la nascita del fondo né l'ingresso della persona nella società che lo gestisce,
quindi vale la più recente delle due.

La cella successiva è un **controllo**: ogni contatore di `Person.csv` viene
confrontato con le righe della sua tabella di dettaglio, e la somma per gruppo
con l'ultima riga dei conteggi cumulati, che contiene i ruoli di qualunque anno.
Le poche differenze che restano sono incoerenze della fonte, sempre di un ruolo
in più nella tabella di dettaglio, che è la più completa.

In [10]:
# 2.4 · l'esperienza anno per anno
persone = db3.select("PersonID").unique()
anno_di = lambda colonna: parse_date(pl.col(colonna)).dt.year()

def leggi(tabella: str, colonne: list[str]) -> pl.DataFrame:
    """Una tabella delle persone, solo per le persone di db3."""
    return nullify(read_raw(cfg, tabella, colonne), MISSING_TOKENS).join(persone, on="PersonID", how="semi")

fondazioni = pl.concat([
    nullify(read_raw(cfg, "Company", ["CompanyID", "YearFounded"]), MISSING_TOKENS)
    .select(pl.col("CompanyID").alias("Entita"), pl.col("YearFounded").cast(pl.Int64, strict=False).alias("Fondazione")),
    nullify(read_raw(cfg, "Investor", ["InvestorID", "YearFounded"]), MISSING_TOKENS)
    .select(pl.col("InvestorID").alias("Entita"), pl.col("YearFounded").cast(pl.Int64, strict=False).alias("Fondazione")),
]).drop_nulls().unique("Entita", keep="first", maintain_order=True)

posizioni = leggi("PersonPositionRelation", ["PersonID", "EntityID", "StartDate"])
seggi = leggi("PersonBoardSeatRelation", ["PersonID", "CompanyID", "StartDate"])
advisory = leggi("PersonAdvisoryRelation", ["PersonID", "EntityID", "StartDate"])
deal = leggi("PersonAffiliatedDealRelation", ["PersonID", "CompanyID", "DealDate"])
fondi = leggi("PersonAffiliatedFundRelation", ["PersonID", "FundID", "InvestorID"])

ingresso = (
    posizioni.with_columns(anno_di("StartDate").alias("_ingresso"))
    .group_by("PersonID", "EntityID").agg(pl.col("_ingresso").min())
)
fondi_datati = (
    fondi.join(nullify(read_raw(cfg, "Fund", ["FundID", "Vintage"]), MISSING_TOKENS)
               .with_columns(pl.col("Vintage").cast(pl.Int64, strict=False)), on="FundID", how="left")
    .join(ingresso, left_on=["PersonID", "InvestorID"], right_on=["PersonID", "EntityID"], how="left")
    .with_columns(pl.max_horizontal("Vintage", "_ingresso").alias("_affiliazione"))
)

FONTI = {
    "posizioni": (posizioni, "EntityID", anno_di("StartDate"), "Posizioni",
                  ["CurrentPositionsCount", "FormerPositionsCount"]),
    "seggi": (seggi, "CompanyID", anno_di("StartDate"), "Seggi",
              ["CurrentBoardSeatsCount", "FormerBoardSeatsCount"]),
    "advisory": (advisory, "EntityID", anno_di("StartDate"), "AltriRuoli",
                 ["CurrentAdvisoryRolesCount", "FormerAdvisoryRolesCount"]),
    "deal": (deal, "CompanyID", anno_di("DealDate"), "AltriRuoli", ["AffiliatedDealsCount"]),
    "fondi": (fondi_datati, "InvestorID", pl.col("_affiliazione"), "AltriRuoli", ["NumberOfAffiliatedFunds"]),
}

def eventi(nome: str, righe: pl.DataFrame, entita: str, anno: pl.Expr, gruppo: str) -> pl.DataFrame:
    """Un evento per ruolo: data vera, altrimenti fondazione dell'entita'.

    Chi non ha nessuna delle due esce con anno 0 e origine "da risolvere": lo
    sistema il blocco qui sotto, che ha bisogno di TUTTI gli eventi della
    persona e quindi non puo' girare qui dentro.
    """
    return (
        righe.join(fondazioni, left_on=entita, right_on="Entita", how="left")
        .select("PersonID",
                pl.coalesce(anno, pl.col("Fondazione"), pl.lit(0)).cast(pl.Int64).alias("anno"),
                pl.lit(gruppo).alias("gruppo"),
                pl.lit(nome).alias("fonte"),
                pl.when(anno.is_not_null()).then(pl.lit("data"))
                .when(pl.col("Fondazione").is_not_null()).then(pl.lit("fondazione"))
                .otherwise(pl.lit("sempre")).alias("origine"))
    )

tutti = pl.concat([eventi(nome, *f[:4]) for nome, f in FONTI.items()])

primo_anno = (
    tutti.filter(pl.col("origine") != "sempre")
    .group_by("PersonID").agg(pl.col("anno").min().alias("_primo"))
)
tutti = (
    tutti.join(primo_anno, on="PersonID", how="left")
    .with_columns(
        pl.when(pl.col("origine") != "sempre").then(pl.col("origine"))
        .when(pl.col("_primo").is_not_null()).then(pl.lit("primo anno"))
        .otherwise(pl.lit("sempre")).alias("origine"),
        pl.when(pl.col("origine") == "sempre")
        .then(pl.col("_primo").fill_null(0))
        .otherwise(pl.col("anno")).cast(pl.Int64).alias("anno"),
    )
    .drop("_primo")
)
del primo_anno

print(f"eventi: {tutti.height:,} per {tutti['PersonID'].n_unique():,} persone")
print(tutti.group_by("gruppo", "origine").len().sort("gruppo", "origine"))

for nome, (righe, *_) in FONTI.items():
    prodotti = tutti.filter(pl.col("fonte") == nome).height
    assert prodotti == righe.height, f"{nome}: {righe.height:,} righe lette ma {prodotti:,} eventi"

GRUPPI = ["Posizioni", "Seggi", "AltriRuoli"]
esperienza = (
    tutti.group_by("PersonID", "anno")
    .agg(*[(pl.col("gruppo") == g).sum().cast(pl.Int64).alias(g) for g in GRUPPI])
    .sort("PersonID", "anno")
    .with_columns(*[pl.col(g).cum_sum().over("PersonID") for g in GRUPPI])
)

esperienza.write_parquet(cfg.interim("esperienza_persona_anno.parquet"))
print(f"esperienza_persona_anno: {esperienza.height:,} righe x {esperienza.width} colonne")
del persone, fondazioni, posizioni, seggi, advisory, deal, fondi, ingresso, fondi_datati
gc.collect()

eventi: 918,698 per 404,468 persone
shape: (9, 3)
┌────────────┬────────────┬────────┐
│ gruppo     ┆ origine    ┆ len    │
│ ---        ┆ ---        ┆ ---    │
│ str        ┆ str        ┆ u32    │
╞════════════╪════════════╪════════╡
│ AltriRuoli ┆ data       ┆ 220567 │
│ AltriRuoli ┆ fondazione ┆ 10059  │
│ AltriRuoli ┆ primo anno ┆ 4566   │
│ Posizioni  ┆ data       ┆ 347122 │
│ Posizioni  ┆ fondazione ┆ 110062 │
│ Posizioni  ┆ primo anno ┆ 18788  │
│ Seggi      ┆ data       ┆ 142653 │
│ Seggi      ┆ fondazione ┆ 52288  │
│ Seggi      ┆ primo anno ┆ 12593  │
└────────────┴────────────┴────────┘


esperienza_persona_anno: 639,681 righe x 5 colonne


0

In [11]:
# 2.5 · controllo dei conteggi di esperienza
CONTATORI = {
    "CurrentPositionsCount": ("PersonPositionRelation", "Yes"),
    "FormerPositionsCount": ("PersonPositionRelation", "No"),
    "CurrentBoardSeatsCount": ("PersonBoardSeatRelation", "Yes"),
    "FormerBoardSeatsCount": ("PersonBoardSeatRelation", "No"),
    "CurrentAdvisoryRolesCount": ("PersonAdvisoryRelation", "Yes"),
    "FormerAdvisoryRolesCount": ("PersonAdvisoryRelation", "No"),
    "AffiliatedDealsCount": ("PersonAffiliatedDealRelation", None),
    "NumberOfAffiliatedFunds": ("PersonAffiliatedFundRelation", None),
}
GRUPPO_DI = {
    "Posizioni": ["CurrentPositionsCount", "FormerPositionsCount"],
    "Seggi": ["CurrentBoardSeatsCount", "FormerBoardSeatsCount"],
    "AltriRuoli": ["CurrentAdvisoryRolesCount", "FormerAdvisoryRolesCount",
                   "AffiliatedDealsCount", "NumberOfAffiliatedFunds"],
}

ultima = (
    pl.read_parquet(cfg.interim("esperienza_persona_anno.parquet"))
    .sort("PersonID", "anno")
    .group_by("PersonID")
    .agg(pl.col(*GRUPPO_DI).last())
)
persone = ultima.select("PersonID")

tabelle = {}
for tabella, stato in CONTATORI.values():
    if tabella not in tabelle:
        colonne = ["PersonID", "IsCurrent"] if stato is not None else ["PersonID"]
        tabelle[tabella] = nullify(read_raw(cfg, tabella, colonne), MISSING_TOKENS).join(persone, on="PersonID", how="semi")
righe = persone
for contatore, (tabella, stato) in CONTATORI.items():
    df = tabelle[tabella] if stato is None else tabelle[tabella].filter(pl.col("IsCurrent") == stato)
    righe = righe.join(df.group_by("PersonID").agg(pl.len().cast(pl.Int64).alias(f"righe_{contatore}")),
                       on="PersonID", how="left")
righe = righe.with_columns(pl.col("^righe_.*$").fill_null(0))

person = nullify(read_raw(cfg, "Person", ["PersonID", "FullName", *CONTATORI]), MISSING_TOKENS).with_columns(
    to_num(c) for c in CONTATORI
)
confronto = persone.join(person, on="PersonID", how="left").join(righe, on="PersonID").join(ultima, on="PersonID")
assenti = confronto["FullName"].null_count()
if assenti:
    print(f"ATTENZIONE: {assenti:,} persone di esperienza non sono in Person.csv")

riepilogo = []
for contatore in CONTATORI:
    vuoto, n = pl.col(contatore).is_null(), pl.col(f"righe_{contatore}")
    diverso = pl.col(contatore).fill_null(0) != n
    riepilogo.append(confronto.select(
        pl.lit(contatore).alias("contatore"),
        (~diverso).sum().alias("uguali"),
        (vuoto & diverso).sum().alias("diversi, contatore vuoto"),
        (~vuoto & diverso).sum().alias("diversi, contatore valorizzato"),
        (vuoto & (n == 0)).sum().alias("vuoti con 0 righe"),
    ))
riepilogo = pl.concat(riepilogo)
with pl.Config(tbl_cols=6, tbl_width_chars=200, tbl_rows=10):
    print(riepilogo)

diversi_contatore = riepilogo.select(pl.col("^diversi.*$")).sum().sum_horizontal().item()
if diversi_contatore == 0:
    print("Contatori: identici per tutte le persone")
else:
    print(f"\nATTENZIONE: {diversi_contatore} contatori diversi dalle righe delle loro tabelle")
    for contatore in CONTATORI:
        qui = confronto.filter(pl.col(contatore).fill_null(0) != pl.col(f"righe_{contatore}"))
        if qui.height:
            with pl.Config(tbl_cols=6, tbl_width_chars=200, fmt_str_lengths=30):
                print(f"\n{contatore}:")
                print(qui.select("PersonID", "FullName", pl.col(contatore).alias("Person.csv"),
                                 pl.col(f"righe_{contatore}").alias("righe nella tabella")))

for g, contatori in GRUPPO_DI.items():
    sbagliate = confronto.filter(pl.sum_horizontal([f"righe_{c}" for c in contatori]) != pl.col(g)).height
    assert sbagliate == 0, f"{g}: {sbagliate:,} persone con l'ultima riga diversa dalle righe delle tabelle"
diversi_gruppo = confronto.filter(pl.any_horizontal([
    pl.sum_horizontal([pl.col(c).fill_null(0) for c in contatori]) != pl.col(g) for g, contatori in GRUPPO_DI.items()
])).height
print(f"\nPer gruppo: l'ultima riga coincide con le tabelle per tutte le {confronto.height:,} persone; "
      f"con Person.csv differisce per {diversi_gruppo} persone (le stesse dei contatori qui sopra)")
del ultima, persone, tabelle, righe, person, confronto, riepilogo
gc.collect()

shape: (8, 5)
┌───────────────────────────┬────────┬──────────────────────────┬────────────────────────────────┬───────────────────┐
│ contatore                 ┆ uguali ┆ diversi, contatore vuoto ┆ diversi, contatore valorizzato ┆ vuoti con 0 righe │
│ ---                       ┆ ---    ┆ ---                      ┆ ---                            ┆ ---               │
│ str                       ┆ u32    ┆ u32                      ┆ u32                            ┆ u32               │
╞═══════════════════════════╪════════╪══════════════════════════╪════════════════════════════════╪═══════════════════╡
│ CurrentPositionsCount     ┆ 404468 ┆ 0                        ┆ 0                              ┆ 176276            │
│ FormerPositionsCount      ┆ 404468 ┆ 0                        ┆ 0                              ┆ 222948            │
│ CurrentBoardSeatsCount    ┆ 404467 ┆ 1                        ┆ 0                              ┆ 321432            │
│ FormerBoardSeatsCount     ┆ 4044

0

In [12]:
# 2.6 · il titolo di studio e il campo di studi
REGOLE_TITOLO = [
    (r"PhD|Doctor|MD|PsyD|DPhil|DC|DDS|DPT|OD|JD", "PhD/Doctorate"),
    (r"MBA|LLM|Master|MSc|MPhil|MPA|MFA|MEng|MAcc|Graduate", "Master's"),
    (r"Bachelor|BSc|BEng|Laurea|BS|BFA|BCom|degree|Business Program|Undergrad", "Bachelor's"),
    (
        r"Certified|Certificat|A Levels|A-Levels|Diplom|DEA|DESS|Dipl\.-Ing|"
        r"Executive Development Program|Executive Education|Executive Education program|"
        r"Executive Program|First Legal State Exam|Legal Practice Course|Vordiplom",
        "Diploma/Certificate",
    ),
]
GERARCHIA = ["Other", "Diploma/Certificate", "Bachelor's", "Master's", "PhD/Doctorate"]

REGOLE_AREA = [
    (r"business|management|bank|invest|financ|marketing|real estate|account|"
     r"Entrepreneur|commerce|econom|actuarial science|private equity", "Economics"),
    (r"engineer|civil|electric|mechanic|electronic|Operations Research|material|"
     r"logistic|Engeneering", "Engineering"),
    (r"statistic|machine learning|Natural Language|robot|technolog|comput|data|"
     r"informatic|Artificial Intelligence|information science|information systems|"
     r"data science|softwar", "IT and Computer Science"),
    (r"law|tax|justice|forensic|legal|jurisprudence|intellectual property", "Law"),
    (r"medic|nursing|pharmac|health|immunolog|neuroscien|genetic|physio", "Health and Medicine"),
    (r"social science|strateg|sociology|psychology|anthropology|international relations|"
     r"polit|government|geography|polic|international|foreign service|social studies|"
     r"criminology|cognitive science|public affairs|urban planning|social work|"
     r"human resource|leadership|foreign", "Social Sciences"),
    (r"natural|biolog|chemistry|physic|environmental science|math|geology|life science|"
     r"zoology|agriculture", "Natural Sciences"),
    (r"humanit|literature|histor|philosoph|language|linguist|english|spanis|american|"
     r"religion|classic|french|theolog|cultural studies|europe|arts|design|music|"
     r"architecture|journalism|media|public relations|advertising|communic|education|"
     r"early childhood|special education|administration", "Humanities and Arts"),
]

studi = nullify(
    read_raw(cfg, "PersonEducationRelation",
             ["PersonID", "Degree", "Major_Concentration", "GraduatingYear", "Institute"]),
    MISSING_TOKENS,
)
studi = studi.with_columns(
    first_match(REGOLE_TITOLO, pl.col("Degree"), pl.lit("Other")).alias("DegreeLevel")
)

titolo, area = pl.col("Degree"), pl.col("Major_Concentration")
studi = studi.with_columns(
    pl.when(area.is_null() & titolo.str.contains("(?i)law").fill_null(False)).then(pl.lit("Law"))
    .when(area.is_null() & titolo.str.contains("(?i)MBA").fill_null(False)).then(pl.lit("Business"))
    .when(area.is_null() & titolo.str.contains("(?i)Medicine").fill_null(False)).then(pl.lit("Medicine"))
    .otherwise(area)
    .alias("Major_Concentration")
)
studi = studi.with_columns(
    pl.when(pl.col("Major_Concentration").is_null())
    .then(None)
    .otherwise(first_match(REGOLE_AREA, pl.col("Major_Concentration"), pl.lit("Other")))
    .alias("Field")
)
print(studi["Field"].value_counts(sort=True))

shape: (10, 2)
┌─────────────────────────┬────────┐
│ Field                   ┆ count  │
│ ---                     ┆ ---    │
│ str                     ┆ u32    │
╞═════════════════════════╪════════╡
│ Economics               ┆ 488278 │
│ null                    ┆ 238415 │
│ Law                     ┆ 153996 │
│ Engineering             ┆ 99971  │
│ Humanities and Arts     ┆ 53061  │
│ Natural Sciences        ┆ 52460  │
│ Social Sciences         ┆ 48137  │
│ Other                   ┆ 45856  │
│ IT and Computer Science ┆ 44208  │
│ Health and Medicine     ┆ 21672  │
└─────────────────────────┴────────┘


In [13]:
# 2.7 · l'istruzione anno per anno
AREE = {
    "Is_Eco": "Economics",
    "Is_Eng": "Engineering",
    "Is_Med": "Health and Medicine",
    "Is_Hum": "Humanities and Arts",
    "Is_IT": "IT and Computer Science",
    "Is_Law": "Law",
    "Is_NS": "Natural Sciences",
    "Is_SS": "Social Sciences",
}

indice_titolo = (
    pl.col("DegreeLevel")
    .replace_strict({nome: i + 1 for i, nome in enumerate(GERARCHIA)}, default=None)
    .cast(pl.Int64)
)
ha_campo = pl.col("Field").is_not_null().sum() > 0

AGGREGAZIONE = [
    *[pl.when(ha_campo).then(pl.col("Field").eq(valore).any()).otherwise(None).alias(flag)
      for flag, valore in AREE.items()],
    pl.col("GraduatingYear").cast(pl.Float64, strict=False).min().alias("Earliest_Year"),
    indice_titolo.max().alias("Highest_Degree"),
    pl.when(pl.col("Institute").is_not_null().sum() > 0)
    .then(pl.col("Institute").drop_nulls().str.join("; "))
    .otherwise(None)
    .alias("Institute"),
]

titoli = (
    studi.join(db3.select("PersonID").unique(), on="PersonID", how="semi")
    .with_row_index("_ordine")
    .with_columns(pl.col("GraduatingYear").cast(pl.Int64, strict=False).fill_null(0).alias("anno_titolo"))
)
punti = titoli.select("PersonID", pl.col("anno_titolo").alias("anno")).unique()
istruzione = (
    punti.join(titoli, on="PersonID")
    .filter(pl.col("anno_titolo") <= pl.col("anno"))
    .sort("_ordine")
    .group_by("PersonID", "anno", maintain_order=True)
    .agg(AGGREGAZIONE)
    .sort("PersonID", "anno")
)
istruzione.write_parquet(cfg.interim("istruzione_persona_anno.parquet"))
print(f"titoli: {titoli.height:,} di {titoli['PersonID'].n_unique():,} persone")
print(f"istruzione_persona_anno: {istruzione.height:,} righe x {istruzione.width} colonne")
del studi, punti, istruzione
gc.collect()

titoli: 292,576 di 173,282 persone
istruzione_persona_anno: 254,196 righe x 13 colonne


0

In [14]:
# 2.8 · controllo dell'istruzione per anno
istruzione = pl.read_parquet(cfg.interim("istruzione_persona_anno.parquet"))
ultima = (
    istruzione.sort("PersonID", "anno")
    .group_by("PersonID")
    .agg(pl.all().exclude("anno").last())
    .sort("PersonID")
)
fotografia = titoli.sort("_ordine").group_by("PersonID", maintain_order=True).agg(AGGREGAZIONE).sort("PersonID")
colonne = fotografia.columns
if not ultima.select(colonne).equals(fotografia):
    for c in colonne[1:]:
        j = fotografia.select("PersonID", c).join(ultima.select("PersonID", c), on="PersonID", suffix="_ultima")
        diverse = j.filter(pl.col(c).ne_missing(pl.col(f"{c}_ultima"))).height
        if diverse:
            print(f"ATTENZIONE {c}: {diverse:,} persone diverse")
assert ultima.select(colonne).equals(fotografia), "l'ultima riga di istruzione non coincide con l'aggregazione di tutti i titoli"
print(f"Identici: per tutte le {fotografia.height:,} persone l'ultima riga coincide con l'aggregazione di tutti i titoli")
cambia = istruzione.group_by("PersonID").agg(pl.len()).filter(pl.col("len") > 1).height
print(f"persone con l'istruzione che cambia nel tempo (piu' di un punto): {cambia:,}")
del istruzione, ultima, fotografia, titoli, AGGREGAZIONE
gc.collect()

Identici: per tutte le 173,282 persone l'ultima riga coincide con l'aggregazione di tutti i titoli
persone con l'istruzione che cambia nel tempo (piu' di un punto): 59,925


0

In [15]:
# 2.9 · i titoli ricavati dal nome della persona
nome = pl.col("PersonName")
ha_phd = nome.str.contains(r"Ph\.?D").fill_null(False)
ha_jd = nome.str.contains(" JD", literal=True).fill_null(False)
ha_md = nome.str.contains(" MD", literal=True).fill_null(False)

db3 = db3.with_columns(ha_phd.alias("Nome_PhD"), ha_jd.alias("Nome_JD"), ha_md.alias("Nome_MD"))
print(f"nomi con Ph.D: {db3['Nome_PhD'].sum():,}  ' JD': {db3['Nome_JD'].sum():,}  ' MD': {db3['Nome_MD'].sum():,}")

nomi con Ph.D: 35,032  ' JD': 801  ' MD': 3,366


In [16]:
# 2.10 · chi e' fondatore
posizioni = nullify(
    read_raw(cfg, "PersonPositionRelation", ["PersonID", "EntityID", "PositionLevel"]),
    MISSING_TOKENS,
).unique(subset=["EntityID", "PersonID"], keep="first", maintain_order=True)

db3 = db3.join(
    posizioni, left_on=["CompanyID", "PersonID"], right_on=["EntityID", "PersonID"], how="left"
)
del posizioni
gc.collect()

qualifica = pl.concat_str(
    [pl.col("FullTitle").fill_null("NA"), pl.col("PositionLevel").fill_null("NA")], separator="; "
)
qualifica = qualifica.str.replace_all(r"(?i)founders?'?s?'? associate", "")
db3 = db3.with_columns(qualifica.str.contains("(?i)founde|founding").alias("IsFounder"))
db3 = db3.drop("PersonName", "FullTitle", "PositionLevel")
print(f"founder: {db3['IsFounder'].sum():,}   IsFounder nullo: {db3['IsFounder'].null_count()}")

founder: 206,117   IsFounder nullo: 0


### Da quando a quando una persona è in azienda

Tre decisioni, e sono quelle che determinano tutte le colonne di team.

**L'ingresso.** Chi è fondatore parte dall'anno di fondazione, anche dove esiste
una data d'ingresso successiva: è la definizione di fondatore. Una data anteriore
alla fondazione viene riportata alla fondazione, perché una presenza non può
cominciare prima che l'azienda esista. Chi **non** è fondatore e non ha una data
d'ingresso non viene invece contato: farlo partire dalla fondazione metterebbe
nel team dei primi anni persone arrivate anni dopo, e quelle persone sono di più
proprio nelle aziende che poi crescono. La riga resta però in `db3`, perché serve
agli attributi del CEO.

**L'uscita.** Una data di uscita mancante diventa l'ultimo anno di vita
dell'azienda, qualunque sia lo stato dichiarato: chi è ancora in carica resta
fino alla fine, e chi non lo è più ma non ha una data di uscita resta anche lui,
perché contare una persona un anno di troppo costa meno che perderla. Il prezzo è
un `Total_People` un po' più alto.

**Il limite.** Nessuna finestra va oltre l'ultimo anno di vita dell'azienda: i
ruoli che cominciano dopo si scartano e le uscite successive si tagliano lì.
Dopo quell'anno la fase 1 non ha nessun dato sull'azienda, quindi quegli anni non
potrebbero dare un valore al target.

L'ultima cella converte le due date in **età dell'azienda**, che è l'unità di
misura del panel, e raddrizza le finestre impossibili.

In [17]:
# 2.11 · l'anno d'ingresso in azienda
db3 = db3.join(vita_azienda, on="CompanyID", how="left")

inizio_fondazione = pl.date(pl.col("YearFounded"), 1, 1)
inizia_prima = pl.col("StartDate").dt.year() < pl.col("YearFounded")

non_datato = pl.col("StartDate").is_null() & ~pl.col("IsFounder")
print(f"non founder senza StartDate, esclusi dalle colonne di team: {db3.select(non_datato.sum()).item():,} su {db3.height:,}")
db3 = db3.with_columns(
    pl.when(pl.col("StartDate").is_null() & pl.col("IsFounder")).then(inizio_fondazione)
    .when(inizia_prima.fill_null(False)).then(inizio_fondazione)
    .otherwise(pl.col("StartDate"))
    .alias("StartDate")
)
print(f"StartDate valorizzate: {db3['StartDate'].is_not_null().sum():,} su {db3.height:,}")

non founder senza StartDate, esclusi dalle colonne di team: 134,894 su 465,861


StartDate valorizzate: 330,967 su 465,861


In [18]:
# 2.12 · l'anno di uscita, e il taglio all'ultimo anno di vita
fine_vita = pl.date(pl.col("UltimoAnno"), 12, 31)
dopo_la_fine = (pl.col("StartDate").dt.year() > pl.col("UltimoAnno")).fill_null(False)
scartati = db3.select(dopo_la_fine.sum()).item()
tagliate = db3.select((pl.col("EndDate").dt.year() > pl.col("UltimoAnno")).sum()).item()
imputate = db3["EndDate"].is_null().sum()

db3 = (
    db3.filter(~dopo_la_fine)
    .with_columns(pl.min_horizontal(pl.col("EndDate").fill_null(fine_vita), fine_vita).alias("EndDate"))
    .drop("IsCurrent", "UltimoAnno")
)
print(f"ruoli iniziati dopo l'ultimo anno di vita, scartati: {scartati:,}")
print(f"EndDate esplicite oltre l'ultimo anno, tagliate    : {tagliate:,}")
print(f"EndDate imputate all'ultimo anno di vita           : {imputate:,}   righe: {db3.height:,}")

ruoli iniziati dopo l'ultimo anno di vita, scartati: 3,574
EndDate esplicite oltre l'ultimo anno, tagliate    : 3,502
EndDate imputate all'ultimo anno di vita           : 379,728   righe: 462,287


In [19]:
# 2.13 · dalle date all'eta' dell'azienda
db3 = db3.with_columns(
    pl.when(pl.col("EndDate") < pl.col("StartDate"))
    .then(pl.col("StartDate"))
    .otherwise(pl.col("EndDate"))
    .alias("EndDate")
).with_columns(
    (pl.col("StartDate").dt.year() - pl.col("YearFounded")).alias("DeltaStart"),
    (pl.col("EndDate").dt.year() - pl.col("YearFounded")).alias("DeltaEnd"),
)

db3 = db3.with_columns(
    pl.when(pl.col("IsFounder")).then(0).otherwise(pl.col("DeltaStart")).alias("DeltaStart")
).drop("StartDate", "EndDate")

db3.write_parquet(cfg.interim("db3.parquet"))
print(f"db3: {db3.height:,} righe x {db3.width} colonne")
print(f"colonne: {db3.columns}")

db3: 462,287 righe x 10 colonne
colonne: ['CompanyID', 'PersonID', 'Gender', 'Nome_PhD', 'Nome_JD', 'Nome_MD', 'IsFounder', 'YearFounded', 'DeltaStart', 'DeltaEnd']


---
## Fase 3 — le colonne di team, anno per anno

Ogni persona viene **espansa** su tutti gli anni della sua finestra, poi le righe
si collassano per (azienda, anno). Ne escono tredici aggregati: quante persone, la
quota di donne, gli otto flag sulle aree di studio, l'anno di laurea medio, il
titolo di studio medio, gli atenei, l'indice di esperienza medio e quanti
fondatori.

**Gli attributi dell'anno.** Prima di aggregare, ogni riga (azienda, anno,
persona) prende l'esperienza e l'istruzione di quella persona **in quell'anno**,
con due join che cercano la riga più recente fra quelle non successive all'anno
della riga. Un titolo preso nel 2018 non alza quindi il titolo di studio nella
riga del 2010. I titoli ricavati dal nome non hanno una data e valgono in tutti
gli anni.

**L'indice di esperienza** è la media di tre conteggi standardizzati, dopo una
trasformazione logaritmica che comprime le code. Media e deviazione standard si
calcolano **a finestra espansiva**: per la riga di un certo anno entrano solo le
coppie (persona, anno) di quell'anno o precedenti. Altrimenti il valore di una
riga del 2005 dipenderebbe anche dalle righe del 2020, e la scala dei primi anni
del panel — quando le persone documentate sono poche — sarebbe quella di
vent'anni dopo. I parametri si salvano, perché li riusa l'indice del CEO.

**Due dettagli delle aggregazioni.** Il denominatore della quota di donne sono le
persone di genere noto: tenere al denominatore chi non ha un genere abbasserebbe
la quota proprio nelle aziende documentate peggio, e dove nessuno ha un genere la
colonna resta vuota. Gli atenei si concatenano scartando i mancanti, così la
stringa non contiene segnaposto.

Il panel è lo **scheletro**, e il team si aggancia ai suoi anni con un left join:
un anno-azienda coperto solo dal team cadrebbe oltre l'ultimo anno di dati, dove
non c'è un target. Dopo il taglio della fase 2 il caso non si presenta, e un
controllo lo garantisce a ogni esecuzione.

In [20]:
# 3.1 · l'espansione a (azienda, anno, persona)
db3 = pl.read_parquet(cfg.interim("db3.parquet"))

espanso = expand_team(db3, min_founding_year=ANNO_MIN_FONDAZIONE)
del db3
gc.collect()
print(f"righe (azienda, anno, persona): {espanso.height:,}")

righe (azienda, anno, persona): 2,340,782


In [21]:
# 3.2 · esperienza e istruzione dell'anno, e l'indice
GRUPPI = ["Posizioni", "Seggi", "AltriRuoli"]
esperienza = pl.read_parquet(cfg.interim("esperienza_persona_anno.parquet"))
istruzione = pl.read_parquet(cfg.interim("istruzione_persona_anno.parquet")).rename({"anno": "anno_istruzione"})

def fotografia(tabella: pl.DataFrame, colonna_anno: str) -> pl.DataFrame:
    return (
        tabella.sort("PersonID", colonna_anno)
        .group_by("PersonID")
        .agg(pl.all().exclude(colonna_anno).last())
    )

espanso = (
    espanso.with_row_index("_riga")
    .with_columns((pl.col("YearFounded") + pl.col("Years")).alias("_anno"))
    .sort("_anno")
    # acceso: la riga della persona con l'anno piu' recente fra quelli <= _anno.
    # spento: l'ultima riga della persona, cioe' il valore alla data di estrazione.
    .pipe(lambda d: (
        d.join_asof(esperienza.sort("anno"), left_on="_anno", right_on="anno",
                    by="PersonID", strategy="backward")
        .join_asof(istruzione.sort("anno_istruzione"), left_on="_anno", right_on="anno_istruzione",
                   by="PersonID", strategy="backward")
        if TEMPORIZZA_PERSONE else
        d.join(fotografia(esperienza, "anno"), on="PersonID", how="left")
        .join(fotografia(istruzione, "anno_istruzione"), on="PersonID", how="left")
    ))
    .sort("_riga")
    .with_columns(pl.col(*GRUPPI).fill_null(0))
    .with_columns(
        pl.when(pl.col("Nome_PhD") | pl.col("Nome_JD") | pl.col("Nome_MD")).then(5)
        .otherwise(pl.col("Highest_Degree")).alias("Highest_Degree"),
        pl.when(pl.col("Nome_JD")).then(True).otherwise(pl.col("Is_Law")).alias("Is_Law"),
        pl.when(pl.col("Nome_MD")).then(True).otherwise(pl.col("Is_Med")).alias("Is_Med"),
    )
)

log1p = {g: (pl.col(g) + 1).log() for g in GRUPPI}

if TEMPORIZZA_PERSONE:
    popolazione = espanso.unique(["PersonID", "_anno"]).sort(["PersonID", "_anno"])
    parametri = pl.concat([
        popolazione.filter(pl.col("_anno") <= Y).select(
            pl.lit(Y, dtype=pl.Int64).alias("_anno"),
            pl.len().alias("_n"),
            *[log1p[g].mean().alias(f"{g}_media") for g in GRUPPI],
            *[log1p[g].std(ddof=1).alias(f"{g}_dev") for g in GRUPPI],
        )
        for Y in sorted(popolazione["_anno"].unique().to_list())
    ])
    del popolazione
else:
    chiave = ["CompanyID", "PersonID"]
    parametri = espanso.unique(chiave).sort(chiave).select(
        *[log1p[g].mean().alias(f"{g}_media") for g in GRUPPI],
        *[log1p[g].std(ddof=1).alias(f"{g}_dev") for g in GRUPPI],
    )
parametri.write_parquet(cfg.interim("parametri_esperienza.parquet"))

COLONNE_PAR = [f"{g}_{s}" for g in GRUPPI for s in ("media", "dev")]
if TEMPORIZZA_PERSONE:
    espanso = espanso.join(parametri.drop("_n"), on="_anno", how="left")
    assert espanso.select(pl.col(COLONNE_PAR[0]).is_null().sum()).item() == 0, "anni senza parametri"
    indice = pl.mean_horizontal(
        [(log1p[g] - pl.col(f"{g}_media")) / pl.col(f"{g}_dev") for g in GRUPPI]
    )
else:
    indice = pl.mean_horizontal(
        [(log1p[g] - parametri[f"{g}_media"][0]) / parametri[f"{g}_dev"][0] for g in GRUPPI]
    )

espanso = (
    espanso.with_columns(indice.alias("WorkExperienceIndex"))
    .drop("_riga", "_anno", *GRUPPI, "Nome_PhD", "Nome_JD", "Nome_MD",
          *[c for c in ("anno", "anno_istruzione") if c in espanso.columns],
          *[c for c in COLONNE_PAR if c in espanso.columns])
)
del esperienza, istruzione
gc.collect()
if TEMPORIZZA_PERSONE:
    print(parametri.head(3))
    print(f"parametri: {parametri.height} anni, popolazione da {parametri['_n'].min():,} a {parametri['_n'].max():,} coppie")
else:
    print(parametri)
print(f"WorkExperienceIndex: media {espanso['WorkExperienceIndex'].mean():.4f} su {espanso.height:,} righe")
print(f"Highest_Degree valorizzato su {espanso['Highest_Degree'].is_not_null().mean():.1%} delle righe")

/tmp/ipykernel_468169/1467395062.py:20: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  d.join_asof(esperienza.sort("anno"), left_on="_anno", right_on="anno",


/tmp/ipykernel_468169/1467395062.py:22: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(istruzione.sort("anno_istruzione"), left_on="_anno", right_on="anno_istruzione",


shape: (3, 8)
┌───────┬───────┬──────────────┬─────────────┬─────────────┬─────────────┬───────────┬─────────────┐
│ _anno ┆ _n    ┆ Posizioni_me ┆ Seggi_media ┆ AltriRuoli_ ┆ Posizioni_d ┆ Seggi_dev ┆ AltriRuoli_ │
│ ---   ┆ ---   ┆ dia          ┆ ---         ┆ media       ┆ ev          ┆ ---       ┆ dev         │
│ i64   ┆ u32   ┆ ---          ┆ f64         ┆ ---         ┆ ---         ┆ f64       ┆ ---         │
│       ┆       ┆ f64          ┆             ┆ f64         ┆ f64         ┆           ┆ f64         │
╞═══════╪═══════╪══════════════╪═════════════╪═════════════╪═════════════╪═══════════╪═════════════╡
│ 2000  ┆ 1991  ┆ 0.612086     ┆ 0.169589    ┆ 0.03619     ┆ 0.292802    ┆ 0.334637  ┆ 0.2221      │
│ 2001  ┆ 5707  ┆ 0.627429     ┆ 0.173372    ┆ 0.037693    ┆ 0.278874    ┆ 0.338146  ┆ 0.220314    │
│ 2002  ┆ 11138 ┆ 0.63492      ┆ 0.178398    ┆ 0.039423    ┆ 0.275174    ┆ 0.342197  ┆ 0.222465    │
└───────┴───────┴──────────────┴─────────────┴─────────────┴─────────────┴───

In [22]:
# 3.3 · i tredici aggregati di team
FLAG_AREE = ["Is_Eco", "Is_Eng", "Is_NS", "Is_Hum", "Is_SS", "Is_Med", "Is_Law", "Is_IT"]

istituto = pl.col("Institute").drop_nulls()

totale = pl.len()
genere_noto = pl.col("Gender").is_not_null().sum()
team = espanso.group_by(["CompanyID", "Years"]).agg(
    totale.alias("Total_People"),
    pl.when(genere_noto > 0)
    .then(pl.col("Gender").eq("Female").sum() / genere_noto * 100)
    .alias("Percent_Females"),
    *[pl.col(c).fill_null(False).any().alias(c) for c in FLAG_AREE],
    pl.col("Earliest_Year").mean().alias("Avg_Earliest_Year"),
    pl.col("Highest_Degree").mean().alias("Highest_Degree_Mean"),
    istituto.unique(maintain_order=True).str.join("; ").alias("Institute"),
    pl.col("WorkExperienceIndex").mean().alias("WorkExp_Idx_Mean"),
    pl.col("IsFounder").sum().alias("Total_Founders"),
)
del espanso
gc.collect()

team = team.with_columns(
    pl.when(pl.col("Institute") == "").then(None).otherwise(pl.col("Institute")).alias("Institute")
)
print(f"team: {team.height:,} anni-azienda x {team.width} colonne")

team: 784,831 anni-azienda x 17 colonne


In [23]:
# 3.4 · il join con lo scheletro
scheletro = pl.read_parquet(cfg.interim("scheletro.parquet"))
team = team.rename({"Years": "Delta"})
print(f"scheletro: {scheletro.height:,} righe   team: {team.height:,} righe")

fuori = team.join(scheletro, on=["CompanyID", "Delta"], how="anti").height
assert fuori == 0, f"{fuori:,} anni-azienda del team fuori dallo scheletro"

panel = scheletro.join(team, on=["CompanyID", "Delta"], how="left").sort(["CompanyID", "Delta"])
del team, scheletro
gc.collect()

panel.write_parquet(cfg.interim("panel_team.parquet"))
print(f"panel: {panel.height:,} righe x {panel.width} colonne")
print(f"righe con dati di team: {panel['Total_People'].is_not_null().sum():,}")
del panel
gc.collect()

scheletro: 907,934 righe   team: 784,831 righe


panel: 907,934 righe x 20 colonne
righe con dati di team: 784,831


0

---
## Fase 4 — i round di finanziamento e gli investitori

I round determinano lo stadio di crescita, quindi il target. Questa fase li
aggrega per azienda-anno, insieme a chi ha messo i soldi.

**Gli investitori** si riducono a sette categorie, e di ognuno interessano due
grandezze: quanti investimenti ha fatto e la dimensione mediana dei round a cui
partecipa. Insieme dicono quanto è grande e attivo chi finanzia. A interruttore
spento sono fotografie alla data di estrazione; a interruttore acceso si
ricostruiscono anno per anno dai deal datati, al prezzo di vedere solo i deal
delle aziende dell'estrazione.

**L'aggregazione per deal** guarda i soli investitori *nuovi* del round, e la
domanda «c'è almeno un nuovo investitore?» ha tre esiti: sì, no, e «non si sa»
quando qualche valore manca. Nel terzo caso l'aggregato resta nullo invece di dire
no. I flag sui lead investor usano lo stesso criterio di incertezza.

**Il capitale raccolto** è la somma degli importi dichiarati, e tratta come zero
un importo ignoto. Poiché in metà delle righe l'importo non è dichiarato, accanto
a `TotalRaised` c'è `UndisclosedAmountShare`, la quota di round dell'anno che non
dichiara quanto ha raccolto: senza di lei «non ha raccolto niente» e «non sappiamo
quanto» sarebbero lo stesso zero. Il panel non contiene nessuna stima del
capitale: gli importi mancanti restano mancanti.

**Il tipo di deal** diventa quattordici flag. Otto di loro non sono colonne finali
ma sono la cascata che produce lo stadio di crescita; due — acceleratori e angel —
si accendono anche quando il tipo di deal non lo dice ma la categoria
dell'investitore sì.

In [24]:
# 4.1 · gli investitori, in sette categorie
CATEGORIA_INVESTITORE = {
    "Venture Capital": "Venture Capital",
    "Corporate Venture Capital": "Venture Capital",
    "Growth/Expansion": "Venture Capital",
    "Not-For-Profit Venture Capital": "Venture Capital",
    "VC-Backed Company": "Venture Capital",
    "Angel (individual)": "Angel",
    "Angel Group": "Angel",
    "Accelerator/Incubator": "Accelerator",
    "Corporation": "Corporate",
    "Corporate Development": "Corporate",
    "PE/Buyout": "Private Equity",
    "Family Office": "Private Equity",
    "PE-Backed Company": "Private Equity",
    "Holding Company": "Private Equity",
    "Merchant Banking Firm": "Private Equity",
    "Mezzanine": "Private Equity",
    "Secondary Buyer": "Private Equity",
    "Other Private Equity": "Private Equity",
    "Special Purpose Acquisition Company (SPAC)": "Private Equity",
    "Fundless Sponsor": "Private Equity",
    "Government": "Public Investor",
    "University": "Public Investor",
    "Sovereign Wealth Fund": "Public Investor",
    "Mutual Fund": "Public Investor",
}
CATEGORIE = {
    "Angel": "Angel",
    "Corporate": "Corporate",
    "VentureCapital": "Venture Capital",
    "Accelerator": "Accelerator",
    "PrivateEquity": "Private Equity",
    "PublicInvestor": "Public Investor",
}
NUMERICHE = ["TotalInvestments", "MedianRoundAmount"]

relazione = nullify(
    read_raw(cfg, "DealInvestorRelation",
             ["DealID", "InvestorID", "InvestorStatus", "IsLeadInvestor"]),
    MISSING_TOKENS,
)
investitori = nullify(
    read_raw(cfg, "Investor", ["InvestorID", "PrimaryInvestorType", *NUMERICHE]), MISSING_TOKENS
).with_columns(to_num(c) for c in NUMERICHE)

relazione = relazione.join(investitori, on="InvestorID", how="left").with_columns(
    pl.col("PrimaryInvestorType")
    .replace_strict(CATEGORIA_INVESTITORE, default="Other")
    .alias("InvestorCategory")
)
del investitori

if TEMPORIZZA_INVESTITORI:
    deal_datati = nullify(read_raw(cfg, "Deal", ["DealID", "DealDate", "DealSize"]), MISSING_TOKENS).with_columns(
        parse_date(pl.col("DealDate")).dt.year().alias("anno"), to_num("DealSize")
    ).drop_nulls("anno")
    storia = (
        nullify(read_raw(cfg, "DealInvestorRelation", ["DealID", "InvestorID"]), MISSING_TOKENS)
        .join(deal_datati.select("DealID", "anno", "DealSize"), on="DealID", how="inner")
    )
    cumulati = (
        storia.select("InvestorID", "anno").unique()
        .join(storia.select("InvestorID", pl.col("anno").alias("_anno_deal"), "DealSize"), on="InvestorID")
        .filter(pl.col("_anno_deal") <= pl.col("anno"))
        .group_by("InvestorID", "anno")
        .agg(pl.len().cast(pl.Float64).alias("TotalInvestments"),
             pl.col("DealSize").median().alias("MedianRoundAmount"))
    )
    base = relazione.drop(*NUMERICHE).join(
        deal_datati.select("DealID", pl.col("anno").alias("_anno_deal")), on="DealID", how="left"
    )
    con_anno = (
        base.drop_nulls("_anno_deal").sort("_anno_deal")
        .join_asof(cumulati.sort("anno"), left_on="_anno_deal", right_on="anno",
                   by="InvestorID", strategy="backward")
        .drop("anno")
    )
    senza_anno = base.filter(pl.col("_anno_deal").is_null()).with_columns(
        *[pl.lit(None, dtype=pl.Float64).alias(c) for c in NUMERICHE]
    )
    relazione = pl.concat([con_anno, senza_anno], how="diagonal_relaxed").drop("_anno_deal")
    del deal_datati, storia, cumulati, base, con_anno, senza_anno
    gc.collect()

print(f"partecipazioni a deal: {relazione.height:,}   deal distinti: {relazione['DealID'].n_unique():,}")
print(f"investitori: {'temporizzati anno per anno' if TEMPORIZZA_INVESTITORI else 'fotografia alla data di estrazione'}")

partecipazioni a deal: 506,641   deal distinti: 278,407
investitori: fotografia alla data di estrazione


In [25]:
# 4.2 · aggregare per deal
nuovo = pl.col("InvestorStatus") == "New Investor"
lead = pl.col("IsLeadInvestor") == "Yes"

condizione = (
    pl.when(nuovo.fill_null(False).any()).then(True)
    .when(nuovo.is_null().any()).then(None)
    .otherwise(False)
)

def media_sui_nuovi(colonna: str) -> pl.Expr:
    """La media calcolata sui soli nuovi investitori, nulla se la condizione e' NA."""
    return if_else_null(condizione, pl.col(colonna).filter(nuovo.fill_null(False)).mean(), None)

def ha_categoria(categoria: str, maschera: pl.Expr) -> pl.Expr:
    """C'e' almeno un investitore di quella categoria, fra quelli selezionati
    dalla maschera? Nullo se la condizione sui nuovi investitori e' NA."""
    appartiene = pl.col("InvestorCategory").filter(maschera.fill_null(False)) == CATEGORIE[categoria]
    return if_else_null(condizione, appartiene.fill_null(False).any(), None)

per_deal = relazione.group_by("DealID").agg(
    nuovo.fill_null(False).sum().alias("TotalInvestors"),
    media_sui_nuovi("TotalInvestments").alias("MeanTotalInvestments"),
    media_sui_nuovi("MedianRoundAmount").alias("MeanMedianRoundAmount"),
    *[ha_categoria(c, nuovo).alias(f"has_{c}") for c in CATEGORIE],
    *[ha_categoria(c, lead).alias(f"has_{c}_Lead") for c in CATEGORIE],
)
del relazione
gc.collect()
print(f"deal con almeno un investitore registrato: {per_deal.height:,}")

deal con almeno un investitore registrato: 278,407


In [26]:
# 4.3 · i deal
COLONNE_DEAL = [
    "CompanyID", "DealID",
    "DealNo",
    "DealDate",
    "DealType",
    "TotalInvestedCapital",
    "CEOPBId",
]

deal = nullify(read_raw(cfg, "Deal", COLONNE_DEAL), MISSING_TOKENS).with_columns(
    pl.col("DealNo").cast(pl.Int64, strict=False),
    parse_date(pl.col("DealDate")).alias("DealDate"),
    to_num("TotalInvestedCapital"),
)
print(f"deal grezzi: {deal.height:,}   senza DealDate: {deal['DealDate'].is_null().sum():,}")

deal = deal.join(
    pl.read_parquet(cfg.interim("db_master_1.parquet")).select(
        "CompanyID", "YearFounded", "OwnershipStatus", "OwnershipStatusDate"
    ),
    on="CompanyID",
    how="left",
)

deal grezzi: 385,481   senza DealDate: 61,169


### Le date dei round che mancano

Un round su sei non ha una data. Quattro passaggi gliene assegnano una, in
quest'ordine:

1. un round di fallimento in un'azienda che risulta fallita prende la data del
   fallimento;
2. un'acquisizione in un'azienda che risulta acquisita prende la data del
   passaggio di proprietà;
3. il **primo** round, se è di un tipo iniziale, va all'anno di fondazione. È
   un'assunzione forte, e decide anche chi entra nel campione, perché schiaccia
   quei round sull'età zero;
4. i round senza data compresi **fra due round datati** si distribuiscono
   uniformemente nell'intervallo, rispettando l'ordine dei round. Si cerca il
   round datato più vicino prima e dopo, non solo quello adiacente, così il
   passaggio funziona anche quando i round senza data sono due o più di fila.

I limiti del quarto passaggio sono sempre round veri: fondazione e ultimo anno
dell'azienda non si usano, quindi un round che non ha nessun round datato accanto
resta senza data. **Quei round non si agganciano a nessun anno e lasciano il
panel**: il blocco 4.6 salva, fuori dal panel, chi li perde e quanti sono, per il
controllo di robustezza. Il bias che introducono va in una direzione sola, un
round non datato non può far salire di stadio.

Un round datato **prima** della fondazione viene invece portato all'anno di
fondazione: sono pochi, e il panel comincia a età zero, prima non esiste una riga
su cui atterrare. Spesso sono eventi veri, che precedono la costituzione legale.

In [27]:
# 4.4 · riparare le date dei deal
FALLIMENTO = ["Bankruptcy: Admin/Reorg", "Bankruptcy: Liquidation", "Out of Business"]
ACQUISITA = ["Acquired/Merged", "Acquired/Merged (Operating Subsidiary)"]
PRIMO_ROUND = ["Accelerator/Incubator", "Angel (individual)", "Grant", "Capitalization",
               "Early Stage VC", "Seed Round", "Spin-Off"]

data_stato = pl.col("OwnershipStatusDate")
mancanti_iniziali = deal["DealDate"].is_null().sum()

deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & pl.col("DealType").is_in(FALLIMENTO)
        & (pl.col("OwnershipStatus") == "Out of Business")
        & data_stato.is_not_null()
    ).then(data_stato).otherwise(pl.col("DealDate")).alias("DealDate")
)
deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & (pl.col("DealType") == "Merger/Acquisition")
        & pl.col("OwnershipStatus").is_in(ACQUISITA)
        & data_stato.is_not_null()
    ).then(data_stato).otherwise(pl.col("DealDate")).alias("DealDate")
)

deal = deal.filter(pl.col("YearFounded") >= ANNO_MIN_FONDAZIONE)
print(f"deal dopo il filtro YearFounded >= {ANNO_MIN_FONDAZIONE}: {deal.height:,}")

deal = deal.with_columns(
    pl.when(
        pl.col("DealDate").is_null()
        & pl.col("DealType").is_in(PRIMO_ROUND)
        & (pl.col("DealNo") == 1)
        & pl.col("YearFounded").is_not_null()
    ).then(pl.date(pl.col("YearFounded"), 1, 1)).otherwise(pl.col("DealDate")).alias("DealDate")
)

deal = deal.sort(["CompanyID", "DealNo"]).with_columns(pl.col("DealDate").dt.year().alias("_anno"))
senza = pl.col("_anno").is_null()
deal = deal.with_columns(
    pl.col("_anno").shift(1).forward_fill().over("CompanyID").alias("_prima"),
    pl.col("_anno").shift(-1).backward_fill().over("CompanyID").alias("_dopo"),
    (~senza).cum_sum().over("CompanyID").alias("_buco"),
)
deal = deal.with_columns(
    senza.cum_sum().over(["CompanyID", "_buco"]).alias("_posizione"),
    senza.sum().over(["CompanyID", "_buco"]).alias("_quanti"),
)
# posizioni equidistanti nel buco: L + (R - L) * i / (k + 1), arrotondato.
passo = (pl.col("_dopo") - pl.col("_prima")) * pl.col("_posizione") / (pl.col("_quanti") + 1)
stima = (pl.col("_prima") + passo + 0.5).floor().cast(pl.Int64)
deal = deal.with_columns(
    pl.when(senza & pl.col("_prima").is_not_null() & pl.col("_dopo").is_not_null())
    .then(pl.date(stima, 1, 1))
    .otherwise(pl.col("DealDate"))
    .alias("DealDate")
).drop("_anno", "_prima", "_dopo", "_buco", "_posizione", "_quanti",
       "OwnershipStatus", "OwnershipStatusDate")

print(f"date mancanti: {mancanti_iniziali:,} -> {deal['DealDate'].is_null().sum():,}")

deal dopo il filtro YearFounded >= 2000: 337,898


date mancanti: 61,169 -> 13,627


In [28]:
# 4.5 · collocare il deal in un anno del panel
anno_deal = pl.col("DealDate").dt.year()

deal = deal.with_columns(
    pl.when(anno_deal.is_null())
    .then(None)
    .otherwise(pl.max_horizontal(anno_deal, pl.col("YearFounded")))
    .alias("Year_Delta"),
)
print(f"deal senza anno, che escono dal panel: {deal['Year_Delta'].is_null().sum():,}")

deal = deal.join(per_deal, on="DealID", how="left")
del per_deal
gc.collect()
deal = nullify(deal, MISSING_TOKENS)

deal senza anno, che escono dal panel: 13,627


In [29]:
# 4.6 · l'elenco delle aziende con round senza data
VC = ["Early Stage VC", "Later Stage VC", "Seed Round"]
senza_anno = pl.col("Year_Delta").is_null()
aziende_round_persi = (
    deal.group_by("CompanyID")
    .agg(
        pl.len().alias("round_totali"),
        senza_anno.sum().alias("round_senza_data"),
        (senza_anno & pl.col("DealType").is_in(VC)).sum().alias("round_vc_senza_data"),
    )
    .filter(pl.col("round_senza_data") > 0)
    .with_columns((pl.col("round_senza_data") == pl.col("round_totali")).alias("perde_tutti"))
    .sort("CompanyID")
)
aziende_round_persi.write_parquet(cfg.interim("aziende_round_senza_data.parquet"))
print(f"aziende con round senza data: {aziende_round_persi.height:,}"
      f"   di cui perdono tutti i round: {aziende_round_persi['perde_tutti'].sum():,}")
print(f"round persi: {aziende_round_persi['round_senza_data'].sum():,}"
      f"   di cui VC: {aziende_round_persi['round_vc_senza_data'].sum():,}")
del aziende_round_persi

aziende con round senza data: 12,042   di cui perdono tutti i round: 757
round persi: 13,627   di cui VC: 3,380


In [30]:
# 4.7 · i quattordici flag sul tipo di deal
PRESEED = ["Accelerator/Incubator", "Angel (individual)", "Grant", "Spin-Off",
           "Equity Crowdfunding", "Product Crowdfunding", "Capitalization"]
USCITA_MA = ["Merger/Acquisition", "Buyout/LBO", "Debt - Acquisition", "Debt - Merger",
             "Merger of Equals", "Investor Buyout by Management", "Corporate Asset Purchase",
             "Reverse Merger"]
USCITA_PUBBLICA = ["IPO", "Secondary Transaction - Open Market",
                   "Secondary Transaction - Stock Distribution",
                   "Public Investment 2nd Offering", "PIPE"]
FALLIMENTI = ["Bankruptcy: Admin/Reorg", "Bankruptcy: Liquidation", "Out of Business",
              "Restart - Angel", "Restart - Early VC", "Restart - Later VC"]
DEBITO = ["Debt - General", "Debt Conversion", "Mezzanine", "Convertible Debt",
          "Debt Refinancing", "Debt - PPP", "Debt Repayment", "Dividend Recapitalization",
          "Exit Financing", "Project Financing", "Share Repurchase", "Leveraged Recapitalization"]
PRIVATE_EQUITY = ["PE Growth/Expansion", "Secondary Transaction - Private", "Corporate",
                  "Platform Creation", "GP Stakes", "General Corporate Purpose", "Capital Spending"]

FLAG_DEAL = {
    "Is_Preseed": PRESEED,
    "Is_Seed": ["Seed Round"],
    "Is_EarlyVC": ["Early Stage VC"],
    "Is_LaterVC": ["Later Stage VC"],
    "Is_MA": USCITA_MA,
    "Is_Public_Exit": USCITA_PUBBLICA,
    "Is_Out": FALLIMENTI,
    "Is_Debt": DEBITO,
    "Is_PE": PRIVATE_EQUITY,
    "Is_Grant": ["Grant"],
    "Is_SpinOff": ["Spin-Off"],
    "Is_CrowdFunding": ["Equity Crowdfunding", "Product Crowdfunding"],
    "Is_Accelerator": ["Accelerator/Incubator"],
    "Is_Angel": ["Angel (individual)"],
}
deal = deal.with_columns(
    *[pl.col("DealType").is_in(v).fill_null(False).alias(f) for f, v in FLAG_DEAL.items()]
)

In [31]:
# 4.8 · aggregare a (azienda, anno)
importo = pl.col("TotalInvestedCapital")

def qualunque(colonna: str) -> pl.Expr:
    """Almeno un True nel gruppo. Un gruppo tutto mancante da' False, non nullo."""
    return pl.col(colonna).fill_null(False).any()

deals_panel = deal.group_by(["CompanyID", "Year_Delta"]).agg(
    pl.len().alias("N_Deal"),
    importo.fill_null(0.0).sum().alias("TotalRaised"),
    importo.is_null().mean().alias("UndisclosedAmountShare"),
    *[qualunque(f).alias(f) for f in FLAG_DEAL if f not in ("Is_Accelerator", "Is_Angel")],
    (qualunque("Is_Accelerator") | qualunque("has_Accelerator")).alias("Is_Accelerator"),
    (qualunque("Is_Angel") | qualunque("has_Angel")).alias("Is_Angel"),
    pl.col("TotalInvestors").fill_null(0).sum().alias("TotalInvestors"),
    pl.col("MeanTotalInvestments").mean().alias("MeanTotalInvestments"),
    pl.col("MeanMedianRoundAmount").mean().alias("MeanMedianRoundAmount"),
    *[qualunque(f"has_{c}").alias(f"has_{c}")
      for c in ("Corporate", "VentureCapital", "PrivateEquity", "PublicInvestor")],
    *[qualunque(f"has_{c}_Lead").alias(f"has_{c}_Lead") for c in CATEGORIE],
    last_non_null("CEOPBId").alias("CEO_ID"),
)
del deal
gc.collect()
print(f"deals_panel: {deals_panel.height:,} anni-azienda con almeno un deal")
print(f"  nel gruppo ad anno nullo (non si agganceranno): "
      f"{deals_panel.filter(pl.col('Year_Delta').is_null()).height:,}")

deals_panel: 286,517 anni-azienda con almeno un deal
  nel gruppo ad anno nullo (non si agganceranno): 12,042


In [32]:
# 4.9 · innestare i deal nel panel
panel = pl.read_parquet(cfg.interim("panel_team.parquet"))

panel = panel.join(deals_panel, on=["CompanyID", "Year_Delta"], how="left").with_columns(
    pl.col("TotalRaised").is_null().cast(pl.Int64).alias("TR_D"),
)
del deals_panel
gc.collect()

panel = nullify(panel, MISSING_TOKENS_WITH_INF)
panel.write_parquet(cfg.interim("panel_deals.parquet"))
print(f"panel: {panel.height:,} righe x {panel.width} colonne")
print(f"TR_D = 1 (anno senza deal): {(panel['TR_D'] == 1).sum():,}")
del panel
gc.collect()

panel: 907,934 righe x 52 colonne
TR_D = 1 (anno senza deal): 635,259


0

---
## Fase 5 — stadi, cumulate e CEO

**I flag diventano cumulativi**: una volta acceso, un flag resta acceso per tutti
gli anni successivi dell'azienda. È così che «ha chiuso un round seed
quest'anno» diventa «ha già chiuso un round seed», e gli stadi diventano monotoni:
un'azienda non retrocede. La maggior parte degli anni-azienda non ha nessun round,
quindi senza la cumulata lo stadio sarebbe vuoto sulla maggior parte delle righe e
il target non starebbe in piedi.

**`GrowthStage`** nasce da una cascata a corto circuito: il primo ramo che
corrisponde vince, quindi l'ordine è parte della definizione. I primi tre rami
sono terminali — fuori mercato, quotata, acquisita — e in essi lo stato di
proprietà concorre con i flag storici, perché esistono aziende chiuse che nessun
deal registra: senza lo stato resterebbero nel panel come se fossero vive. Lo
stato entra solo nell'anno della propria data, che per fallimenti e acquisizioni è
la data dell'evento. I rami successivi ordinano gli stadi di crescita dal più
avanzato al più iniziale.

**Le cumulate.** Il numero di round e di investitori si accumula; le due grandezze
degli investitori diventano medie cumulate **pesate** per quanti nuovi investitori
sono entrati in ciascun anno, così un round con dieci investitori conta dieci
volte un round con uno. Un anno senza round ha raccolto zero, e non ha nemmeno
importi non dichiarati.

In [33]:
# 5.1 · i flag diventano cumulativi
FLAG_CUMULATIVI = [
    "Is_Preseed", "Is_Seed", "Is_EarlyVC", "Is_LaterVC", "Is_MA", "Is_Public_Exit",
    "Is_Out", "Is_Debt", "Is_PE", "Is_Grant", "Is_SpinOff", "Is_CrowdFunding",
    "Is_Accelerator", "Is_Angel",
    "has_Corporate", "has_VentureCapital", "has_PrivateEquity", "has_PublicInvestor",
    "has_Angel_Lead", "has_Corporate_Lead", "has_VentureCapital_Lead",
    "has_Accelerator_Lead", "has_PrivateEquity_Lead", "has_PublicInvestor_Lead",
]

panel = pl.read_parquet(cfg.interim("panel_deals.parquet")).sort(["CompanyID", "Year_Delta"])
panel = panel.with_columns(cumulative_any(pl.col(c)).over("CompanyID").alias(c) for c in FLAG_CUMULATIVI)
print(f"flag resi cumulativi: {len(FLAG_CUMULATIVI)}")

flag resi cumulativi: 24


In [34]:
# 5.2 · GrowthStage, la cascata degli stadi
stato = pl.col("OwnershipStatus")
out, pubblica, ma = pl.col("Is_Out"), pl.col("Is_Public_Exit"), pl.col("Is_MA")
later, pe = pl.col("Is_LaterVC"), pl.col("Is_PE")
early, seed, preseed = pl.col("Is_EarlyVC"), pl.col("Is_Seed"), pl.col("Is_Preseed")
non_terminale = ~ma & ~pubblica & ~out

panel = panel.with_columns(
    pl.when((stato == "Out of Business") | out).then(pl.lit("Out"))
    .when(stato.is_in(["Publicly Held", "In IPO Registration"]) | pubblica).then(pl.lit("Exit_Public"))
    .when(stato.is_in(["Acquired/Merged", "Acquired/Merged (Operating Subsidiary)"]) | ma).then(pl.lit("Exit_M&A"))
    .when((later | pe) & non_terminale).then(pl.lit("LaterVC_or_Other"))
    .when(early & ~later & non_terminale & ~pe).then(pl.lit("EarlyVC"))
    .when(seed & ~early & ~later & non_terminale & ~pe).then(pl.lit("Seed"))
    .when(preseed & ~seed & ~early & ~later & non_terminale & ~pe).then(pl.lit("Preseed"))
    .otherwise(None)
    .alias("GrowthStage")
)
print(panel["GrowthStage"].value_counts(sort=True))

shape: (8, 2)
┌──────────────────┬────────┐
│ GrowthStage      ┆ count  │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ null             ┆ 240900 │
│ Preseed          ┆ 217654 │
│ EarlyVC          ┆ 154334 │
│ LaterVC_or_Other ┆ 112010 │
│ Seed             ┆ 78386  │
│ Exit_M&A         ┆ 57028  │
│ Out              ┆ 35222  │
│ Exit_Public      ┆ 12400  │
└──────────────────┴────────┘


In [35]:
# 5.3 · gli anni senza deal, e le cumulate
panel = panel.with_columns(
    pl.when(pl.col("TR_D") == 1).then(0.0).otherwise(pl.col("TotalRaised")).alias("TotalRaised"),
    pl.when(pl.col("TR_D") == 1).then(0.0)
    .otherwise(pl.col("UndisclosedAmountShare")).alias("UndisclosedAmountShare"),
)

# I nuovi investitori dell'anno si mettono da parte prima di sovrascrivere la
# colonna con la propria cumulata: sono il peso delle medie ponderate qui sotto,
# e polars valuta le espressioni di un with_columns sul frame di partenza.
panel = panel.with_columns(pl.col("TotalInvestors").fill_null(0).alias("NewInvestors"))

panel = panel.with_columns(
    cum_sum_null(pl.col("N_Deal").fill_null(0)).over("CompanyID").alias("N_Deal"),
    cum_sum_null(pl.col("NewInvestors").fill_null(0)).over("CompanyID").alias("TotalInvestors"),
)
print(f"N_Deal massimo: {panel['N_Deal'].max()}   TotalInvestors massimo: {panel['TotalInvestors'].max()}")

N_Deal massimo: 32   TotalInvestors massimo: 191


In [36]:
# 5.4 · le medie ponderate cumulate
PONDERATE = ["MeanTotalInvestments", "MeanMedianRoundAmount"]

panel = panel.with_columns(
    weighted_cumulative(c, "NewInvestors", ["CompanyID"]).alias(f"{c}_cum") for c in PONDERATE
)
panel = panel.drop(*PONDERATE, "NewInvestors")

### Il CEO

Del CEO entrano nel panel tre attributi: genere, indice di esperienza e titolo di
studio più alto. Il CEO è dichiarato **dai deal**, quindi solo negli anni in cui
c'è stato un round: si riporta in avanti, chi era CEO all'ultimo round lo resta
finché non ne arriva un altro.

Questo lascia due problemi, e il blocco li affronta entrambi.

**Prima del primo round il CEO non esiste**, e sono proprio gli anni iniziali,
quelli da cui i modelli leggono le feature. Si riempie con i ruoli da CEO del
board team, ma solo dove l'attribuzione è verificabile, perché quel titolo è una
fotografia alla data di estrazione e proiettarlo all'indietro alla cieca direbbe
«era già CEO» di chi lo è diventato dopo. Le condizioni sono tutte necessarie:

- un solo ruolo da CEO copre quell'anno, così non c'è niente da arbitrare;
- il titolo è insieme di fondatore e di CEO, e per un fondatore «da quando è in
  azienda» coincide con l'anno di fondazione;
- la data d'ingresso è dichiarata, non imputata;
- è la stessa persona che il primo round conferma CEO, che fa da riscontro
  indipendente;
- nessun altro ha un ruolo da CEO cominciato prima di lui.

**Quando un nuovo CEO entra fra un round e l'altro**, il riporto in avanti
continuerebbe a trascinare il vecchio fino al round successivo, attribuendo la
persona sbagliata. Se un ruolo del board comincia, con data dichiarata, dopo
l'ultima dichiarazione dei deal ed è un'altra persona, vince lui.

Resta un limite, da dichiarare: un co-fondatore può essere stato CTO e diventato
CEO più tardi, e la riga direbbe comunque «Co-Founder & CEO» dalla stessa data. Il
riscontro del round delimita **chi**, la fondazione delimita **da quando era in
azienda**, ma l'incertezza sull'etichetta dentro quella finestra non è eliminabile
con questi dati.

Gli attributi si leggono poi come per il team: i ruoli iniziati entro quell'anno e
il titolo conseguito entro quell'anno, con gli stessi parametri di
standardizzazione.

In [37]:
# 5.5 · gli attributi del CEO
CEO_VARS = ["Gender"]

panel = panel.sort("CompanyID", "Year_Delta").with_columns(
    pl.when(pl.col("CEO_ID").is_not_null()).then(pl.col("Year_Delta")).otherwise(None)
      .fill_null(strategy="forward").over("CompanyID").alias("_anno_dich"),
)
_primo = (
    panel.filter(pl.col("CEO_ID").is_not_null())
    .group_by("CompanyID")
    .agg(pl.col("Year_Delta").min().alias("_primo_anno"),
         pl.col("CEO_ID").sort_by("Year_Delta").first().alias("_primo_ceo"))
)
panel = (
    panel.join(_primo, on="CompanyID", how="left")
    .with_columns(pl.col("CEO_ID").fill_null(strategy="forward").over("CompanyID"))
)

ruoli_ceo = pl.read_parquet(cfg.interim("ruoli_ceo.parquet"))
_attivi = (
    panel.select("CompanyID", "Year_Delta")
    .join_where(
        ruoli_ceo,
        pl.col("CompanyID") == pl.col("CompanyID_right"),
        pl.col("Year_Delta") >= pl.col("da"),
        pl.col("Year_Delta") <= pl.col("a"),
    )
    .group_by("CompanyID", "Year_Delta")
    .agg(pl.col("PersonID").n_unique().alias("_n"),
         pl.col("PersonID").first().alias("_board"),
         pl.col("sv").first().alias("_sv"),
         pl.col("is_founder").first().alias("_f"))
    .filter(pl.col("_n") == 1)
)
_prima = (
    ruoli_ceo.filter(pl.col("sv").is_not_null())
    .join(ruoli_ceo.filter(pl.col("sv").is_not_null()), on="CompanyID")
    .filter((pl.col("PersonID") != pl.col("PersonID_right")) & (pl.col("sv_right") < pl.col("sv")))
    .select("CompanyID", pl.col("PersonID").alias("_board")).unique()
    .with_columns(pl.lit(True).alias("_scartata"))
)
panel = (
    panel.join(_attivi, on=["CompanyID", "Year_Delta"], how="left")
    .join(_prima, on=["CompanyID", "_board"], how="left")
)

# Le cinque condizioni del riempimento, tutte necessarie.
_riempibile = (
    pl.col("CEO_ID").is_null()
    & pl.col("_primo_anno").is_not_null() & (pl.col("Year_Delta") < pl.col("_primo_anno"))
    & (pl.col("_n") == 1) & pl.col("_f") & pl.col("_sv").is_not_null()
    & (pl.col("_board") == pl.col("_primo_ceo"))
    & pl.col("_scartata").is_null()
)
# Un nuovo CEO, con data dichiarata, dopo l'ultima dichiarazione dei deal.
_da_correggere = (
    pl.col("CEO_ID").is_not_null() & (pl.col("_n") == 1) & pl.col("_sv").is_not_null()
    & (pl.col("_sv") > pl.col("_anno_dich")) & (pl.col("_board") != pl.col("CEO_ID"))
)
_riempite = panel.select(_riempibile.sum()).item()
_corrette = panel.select(_da_correggere.sum()).item()
panel = panel.with_columns(
    pl.when(_riempibile).then(pl.col("_board"))
    .when(_da_correggere).then(pl.col("_board"))
    .otherwise(pl.col("CEO_ID")).alias("CEO_ID")
).drop("_n", "_board", "_sv", "_f", "_scartata", "_primo_anno", "_primo_ceo", "_anno_dich")
del ruoli_ceo, _attivi, _prima, _primo
gc.collect()
print(f"CEO riempiti prima del primo round: {_riempite:,}   attribuzioni corrette: {_corrette:,}")

ceo = (
    pl.read_parquet(cfg.interim("db3.parquet"))
    .select("CompanyID", "PersonID", *CEO_VARS, "Nome_PhD", "Nome_JD", "Nome_MD")
    .rename({c: f"{c}_CEO" for c in CEO_VARS})
    .with_columns(pl.lit(True).alias("_ceo_nel_team"),
                  (pl.col("Nome_PhD") | pl.col("Nome_JD") | pl.col("Nome_MD")).alias("_ceo_dottore"))
    .drop("Nome_PhD", "Nome_JD", "Nome_MD")
)
panel = panel.join(
    ceo, left_on=["CompanyID", "CEO_ID"], right_on=["CompanyID", "PersonID"], how="left"
)

GRUPPI = ["Posizioni", "Seggi", "AltriRuoli"]
parametri = pl.read_parquet(cfg.interim("parametri_esperienza.parquet"))
esperienza_ceo = pl.read_parquet(cfg.interim("esperienza_persona_anno.parquet"))
istruzione_ceo = (
    pl.read_parquet(cfg.interim("istruzione_persona_anno.parquet"),
                    columns=["PersonID", "anno", "Highest_Degree"])
    .rename({"anno": "anno_istruzione"})
)
ultima_riga = lambda t, a: t.sort("PersonID", a).group_by("PersonID").agg(pl.all().exclude(a).last())
panel = (
    panel.with_row_index("_riga")
    .sort("Year_Delta")
    .pipe(lambda d: (
        d.join_asof(esperienza_ceo.sort("anno"), left_on="Year_Delta", right_on="anno",
                    by_left="CEO_ID", by_right="PersonID", strategy="backward")
        .join_asof(istruzione_ceo.sort("anno_istruzione"), left_on="Year_Delta",
                   right_on="anno_istruzione", by_left="CEO_ID", by_right="PersonID", strategy="backward")
        if TEMPORIZZA_PERSONE else
        d.join(ultima_riga(esperienza_ceo, "anno"), left_on="CEO_ID", right_on="PersonID", how="left")
        .join(ultima_riga(istruzione_ceo, "anno_istruzione"), left_on="CEO_ID", right_on="PersonID", how="left")
    ))
    .sort("_riga")
    .with_columns(pl.col(*GRUPPI).fill_null(0))
)

COLONNE_PAR = [f"{g}_{s}" for g in GRUPPI for s in ("media", "dev")]
if TEMPORIZZA_PERSONE:
    panel = (
        panel.sort("Year_Delta")
        .join(parametri.drop("_n"), left_on="Year_Delta", right_on="_anno", how="left")
        .with_columns(pl.col(COLONNE_PAR).fill_null(strategy="forward"))
        .with_columns(pl.col(COLONNE_PAR).fill_null(strategy="backward"))
        .sort("_riga")
    )
    assert panel.select(pl.col(COLONNE_PAR[0]).is_null().sum()).item() == 0, "anni senza parametri"
    indice_ceo = pl.mean_horizontal(
        [((pl.col(g) + 1).log() - pl.col(f"{g}_media")) / pl.col(f"{g}_dev") for g in GRUPPI]
    )
else:
    indice_ceo = pl.mean_horizontal(
        [((pl.col(g) + 1).log() - parametri[f"{g}_media"][0]) / parametri[f"{g}_dev"][0] for g in GRUPPI]
    )

panel = panel.with_columns(
    pl.when(pl.col("_ceo_nel_team")).then(indice_ceo).alias("WorkExperienceIndex_CEO"),
    pl.when(pl.col("_ceo_nel_team"))
    .then(pl.when(pl.col("_ceo_dottore")).then(5).otherwise(pl.col("Highest_Degree")))
    .alias("Highest_Degree_CEO"),
)
panel = panel.drop("_riga", *GRUPPI, "Highest_Degree", "_ceo_nel_team", "_ceo_dottore", "CEO_ID",
                   *[c for c in ("anno", "anno_istruzione") if c in panel.columns],
                   *[c for c in COLONNE_PAR if c in panel.columns])
del ceo, parametri, esperienza_ceo, istruzione_ceo
gc.collect()
print(f"righe con attributi del CEO: {panel['Gender_CEO'].is_not_null().sum():,}")

CEO riempiti prima del primo round: 71,600   attribuzioni corrette: 3,491


/tmp/ipykernel_468169/3322054691.py:96: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  d.join_asof(esperienza_ceo.sort("anno"), left_on="Year_Delta", right_on="anno",


/tmp/ipykernel_468169/3322054691.py:98: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  .join_asof(istruzione_ceo.sort("anno_istruzione"), left_on="Year_Delta",


righe con attributi del CEO: 710,455


In [38]:
# 5.6 · paese, settore ed eta'
panel = panel.join(
    pl.read_parquet(cfg.interim("db_master_1.parquet")).select(
        "CompanyID", "HQCountry", "PrimaryIndustrySector"
    ),
    on="CompanyID",
    how="left",
)
righe_attese = panel.height

panel = panel.with_columns(pl.col("Delta").alias("Age")).drop(
    "OwnershipStatus", "Delta", "TR_D",
)
assert panel.height == righe_attese, "il join con l'anagrafica ha moltiplicato le righe"
panel.write_parquet(cfg.interim("panel_finale.parquet"))
print(f"panel: {panel.height:,} righe x {panel.width} colonne")
del panel
gc.collect()

panel: 907,934 righe x 55 colonne


0

---
## Fase 6 — i gruppi di stadio e il troncamento

I sette valori di `GrowthStage` si raggruppano in quattro: `Early`, `Later`,
`Exit`, `Out`.

**Lo stadio futuro si calcola prima del troncamento**, e l'ordine fra i due
blocchi è un vincolo: calcolandolo dopo, un'uscita non sarebbe più raggiungibile
come stadio futuro, che è esattamente il senso della colonna. Per ogni riga si
cerca il primo gruppo successivo diverso da quello corrente, e a che distanza in
anni. Le aziende che non lasciano mai il loro gruppo prendono un segnaposto,
sostituito alla fine dal gruppo corrente, e come distanza gli anni che restano.

**Il troncamento** elimina, per ogni azienda, la prima riga terminale e tutto
quello che segue. Togliere anche l'anno dell'uscita non è una dimenticanza: è la
condizione perché il target sia corretto. L'esito resta nello stadio futuro,
calcolato prima, e l'ultima riga rimasta dice «sta per uscire, fra N anni».
Tenendo la riga dell'uscita, l'anno in cui si legge il target si sposterebbe in
avanti e cadrebbe proprio lì, dove lo stadio futuro punta alle righe successive
all'uscita, e centinaia di aziende risulterebbero non uscite.

Cadono così anche le poche righe non terminali che seguono un'uscita — aziende
date per fallite che prendono un altro round, o acquisite che continuano a
raccogliere. È coerente con il trattare l'uscita come uno stato assorbente, e va
dichiarato.

Le regole di questa fase sono state ricostruite a partire dallo schema del panel
di riferimento e verificate riga per riga contro di esso: reggono su tutti i casi
osservati, ma una combinazione mai comparsa lì potrebbe non essere trattata come
si aspetta.

In [39]:
# 6.1 · dai sette stadi ai quattro gruppi
GRUPPO_STADIO = {
    "Preseed": "Early",
    "Seed": "Early",
    "EarlyVC": "Early",
    "LaterVC_or_Other": "Later",
    "Out": "Out",
    "Exit_M&A": "Exit",
    "Exit_Public": "Exit",
}
GRUPPI_TERMINALI = ["Out", "Exit"]

panel = pl.read_parquet(cfg.interim("panel_finale.parquet")).sort(["CompanyID", "Year_Delta"])
panel = panel.with_columns(
    pl.col("GrowthStage").replace_strict(GRUPPO_STADIO, default=None).alias("GrowthStageGroup")
).drop("GrowthStage")
print(panel["GrowthStageGroup"].value_counts(sort=True))

shape: (5, 2)
┌──────────────────┬────────┐
│ GrowthStageGroup ┆ count  │
│ ---              ┆ ---    │
│ str              ┆ u32    │
╞══════════════════╪════════╡
│ Early            ┆ 450374 │
│ null             ┆ 240900 │
│ Later            ┆ 112010 │
│ Exit             ┆ 69428  │
│ Out              ┆ 35222  │
└──────────────────┴────────┘


In [40]:
# 6.2 · lo stadio futuro
panel = next_different(
    panel, "GrowthStageGroup", ["CompanyID"], "GrowthNextStageGroup", "TimeNextStageGroup"
)

righe_rimanenti = pl.len().over("CompanyID") - pl.int_range(pl.len()).over("CompanyID") - 1
# Le due espressioni stanno nello stesso with_columns apposta: la seconda legge
# GrowthNextStageGroup prima che la prima lo riempia.
panel = panel.with_columns(
    pl.col("GrowthNextStageGroup").fill_null("Stay").alias("GrowthNextStageGroup"),
    pl.when(pl.col("GrowthNextStageGroup").is_null())
    .then(righe_rimanenti)
    .otherwise(pl.col("TimeNextStageGroup"))
    .alias("TimeNextStageGroup"),
)
print(panel["GrowthNextStageGroup"].value_counts(sort=True))

shape: (5, 2)
┌──────────────────────┬────────┐
│ GrowthNextStageGroup ┆ count  │
│ ---                  ┆ ---    │
│ str                  ┆ u32    │
╞══════════════════════╪════════╡
│ Stay                 ┆ 611703 │
│ Later                ┆ 119304 │
│ Out                  ┆ 111613 │
│ Exit                 ┆ 64969  │
│ Early                ┆ 345    │
└──────────────────────┴────────┘


In [41]:
# 6.3 · il troncamento all'uscita
prima = panel.height
# Vale 1 dalla prima riga terminale in poi: si tengono solo le righe a 0, quindi
# anche la riga dell'uscita esce dal panel.
raggiunto_terminale = (
    pl.col("GrowthStageGroup")
    .is_in(GRUPPI_TERMINALI)
    .fill_null(False)
    .cast(pl.Int8)
    .cum_max()
    .over("CompanyID")
)
panel = panel.filter(raggiunto_terminale == 0)
print(f"righe: {prima:,} -> {panel.height:,}  (attese 802.148)")
print(f"aziende: {panel['CompanyID'].n_unique():,}  (attese 116.312)")

righe: 907,934 -> 802,148  (attese 802.148)
aziende: 116,312  (attese 116.312)


---
## Fase 7 — i concorrenti

Tre colonne — quanti concorrenti, quanti nello stesso paese, la similarità media —
più `N_Similar`, il numero di aziende simili su cui quella media è calcolata.
Esistono in una sola versione, e il contenuto lo decide `TEMPORIZZA_COMPETITOR`:

- **acceso**: si contano solo i concorrenti *vivi in quell'anno*, e la coppia
  entra solo se della controparte si conosce la finestra di vita, che si legge
  soltanto da `Company.csv`. È la versione temporalmente pulita, e il prezzo è che
  sopravvive circa un decimo delle coppie dichiarate;
- **spento**: ogni informazione disponibile, senza finestra e senza filtro sulla
  controparte, comprese le aziende simili fuori estrazione. È la baseline con il
  look-ahead.

`Company.csv` si rilegge per intero, non solo le coorti del panel: un concorrente
può essere più vecchio dell'anno minimo e va comunque considerato vivo.
L'assunzione più pesante di questa fase è che **l'ultimo anno con dati valga come
«era ancora viva»**: non è una data di chiusura, e un'azienda ben coperta risulta
viva più a lungo di una coperta male, quindi il conteggio sovrappesa le grandi e
le ben documentate. Si tiene perché quell'anno è la fine della vita in tutta la
pipeline, e cambiarlo solo qui introdurrebbe un'incoerenza peggiore.

Due insiemi diversi di coppie: *simili* per la media di similarità, *concorrenti*
per i conteggi. La relazione è orientata e reciproca in una minoranza dei casi: si
conta chi l'azienda dichiara, non chi dichiara lei. Il paese della controparte si
legge dalla tabella delle relazioni e non dall'anagrafica, così resta calcolabile
anche quando la controparte è fuori estrazione.

`N_Similar` esiste perché uno zero nella similarità media è ambiguo: può voler
dire «nessun comparabile» oppure «comparabili per niente simili». È additiva:
eliminarla riporta allo schema precedente.

L'ultimo blocco innesta le colonne, sostituisce il segnaposto dello stadio futuro
e **rinumera le aziende**, che è l'ultima operazione della pipeline: da lì in
avanti gli identificativi originali non esistono più.

In [42]:
# 7.1 · la finestra di vita di ogni azienda
COLONNE_VITA = [*COMPANY_DATE_COLUMNS, "FiscalPeriod", "YearFounded", "HQCountry"]
tutte = nullify(read_raw(cfg, "Company", ["CompanyID", *COLONNE_VITA]), MISSING_TOKENS)

trimestre_v = pl.col("FiscalPeriod").str.extract(r"TTM (\d)Q\d{4}", 1).cast(pl.Int64, strict=False)
anno_v = pl.col("FiscalPeriod").str.slice(-4).cast(pl.Int64, strict=False)
tutte = tutte.with_columns(
    *[parse_date(pl.col(c)).alias(c) for c in COMPANY_DATE_COLUMNS],
    pl.date(anno_v, trimestre_v * 3, 30).alias("FiscalDate"),
    pl.col("YearFounded").cast(pl.Int64, strict=False),
)

vita = (
    tutte.with_columns(
        pl.max_horizontal([pl.col(c).dt.year() for c in [*COMPANY_DATE_COLUMNS, "FiscalDate"]]).alias("MaxYear")
    )
    .select("CompanyID", "YearFounded", "MaxYear", "HQCountry")
    .filter(pl.col("YearFounded").is_not_null() & pl.col("MaxYear").is_not_null())
)
del tutte
gc.collect()
print(f"aziende con una finestra di vita utilizzabile: {vita.height:,} su 134.355")

aziende con una finestra di vita utilizzabile: 126,817 su 134.355


In [43]:
# 7.2 · le coppie azienda / azienda simile
simili_grezze = nullify(
    read_raw(cfg, "CompanySimilarRelation",
             ["CompanyID", "SimilarCompanyID", "SimilarityScore", "IsCompetitor",
              "SimilarCompanyHQCountry"]),
    MISSING_TOKENS,
).with_columns(to_num("SimilarityScore"))

id_panel = panel.select("CompanyID").unique().to_series()
simili_grezze = simili_grezze.filter(pl.col("CompanyID").is_in(id_panel.implode()))

simili_grezze = (
    simili_grezze
    .join(vita.select("CompanyID", pl.col("HQCountry").alias("_paese_proprio")),
          on="CompanyID", how="left")
    .with_columns(
        (pl.col("_paese_proprio") == pl.col("SimilarCompanyHQCountry")).alias("_stesso_paese")
    )
    .drop("_paese_proprio", "SimilarCompanyHQCountry")
)

simili = simili_grezze.select("CompanyID", "SimilarCompanyID", "SimilarityScore")
concorrenti = simili_grezze.filter(pl.col("IsCompetitor") == "Yes").select(
    "CompanyID", "SimilarCompanyID", "SimilarityScore", "_stesso_paese"
)

if TEMPORIZZA_COMPETITOR:
    finestra = vita.select(
        pl.col("CompanyID").alias("SimilarCompanyID"),
        pl.col("YearFounded").alias("YF"),
        pl.col("MaxYear").alias("MY"),
    )
    dichiarate = (concorrenti.height, simili.height)
    simili = simili.join(finestra, on="SimilarCompanyID", how="inner")
    concorrenti = concorrenti.join(finestra, on="SimilarCompanyID", how="inner")
    del finestra
    print(f"coppie dichiarate           : concorrenti {dichiarate[0]:,}   simili {dichiarate[1]:,}")
    print(f"coppie con finestra di vita : concorrenti {concorrenti.height:,}   simili {simili.height:,}")
else:
    print(f"coppie usate (nessun filtro): concorrenti {concorrenti.height:,}   simili {simili.height:,}")

del simili_grezze, vita
gc.collect()

coppie dichiarate           : concorrenti 116,081   simili 1,160,950
coppie con finestra di vita : concorrenti 27,729   simili 208,372


0

In [44]:
# 7.3 · chi era attivo in quale anno
anni_panel = panel.select("CompanyID", "Year_Delta").unique()

if TEMPORIZZA_COMPETITOR:
    # Join di disuguaglianza: il prodotto cartesiano non starebbe in memoria.
    attivi_concorrenti = active_pairs(anni_panel, concorrenti, ["_stesso_paese"])
    attivi_simili = active_pairs(anni_panel, simili, ["SimilarityScore"])
    print(f"coppie concorrente-anno attive: {attivi_concorrenti.height:,}")
    print(f"coppie simile-anno attive     : {attivi_simili.height:,}")
else:
    attivi_concorrenti = attivi_simili = None
    print("TEMPORIZZA_COMPETITOR spento: nessun taglio per anno")

coppie concorrente-anno attive: 190,601
coppie simile-anno attive     : 904,351


In [45]:
# 7.4 · le tre colonne competitor, e il denominatore
if TEMPORIZZA_COMPETITOR:
    stat_concorrenti = attivi_concorrenti.group_by("CompanyID", "Year_Delta").agg(
        pl.col("SimilarCompanyID").n_unique().alias("N_Competitors"),
        pl.col("_stesso_paese").drop_nulls().sum().cast(pl.Int64).alias("Same_Country"),
    )
    stat_simili = attivi_simili.group_by("CompanyID", "Year_Delta").agg(
        pl.col("SimilarityScore").mean().alias("SimilarityScoreMean"),
        pl.col("SimilarCompanyID").n_unique().alias("N_Similar"),
    )
else:
    stat_concorrenti = anni_panel.join(
        concorrenti.group_by("CompanyID").agg(
            pl.col("SimilarCompanyID").n_unique().alias("N_Competitors"),
            pl.col("_stesso_paese").drop_nulls().sum().cast(pl.Int64).alias("Same_Country"),
        ),
        on="CompanyID", how="inner",
    )
    stat_simili = anni_panel.join(
        simili.group_by("CompanyID").agg(
            pl.col("SimilarityScore").mean().alias("SimilarityScoreMean"),
            pl.col("SimilarCompanyID").n_unique().alias("N_Similar"),
        ),
        on="CompanyID", how="inner",
    )

del attivi_concorrenti, attivi_simili, concorrenti, simili
gc.collect()
print(f"anni-azienda con almeno un concorrente: {stat_concorrenti.height:,}")

anni-azienda con almeno un concorrente: 98,409


In [46]:
# 7.5 · innestare, chiudere il target, rinumerare
finale = (
    panel
    .join(stat_concorrenti, on=["CompanyID", "Year_Delta"], how="left")
    .join(stat_simili, on=["CompanyID", "Year_Delta"], how="left")
    .with_columns(
        # Nessun concorrente significa zero concorrenti: sono conteggi.
        pl.col("N_Competitors", "Same_Country", "N_Similar").fill_null(0),
        pl.col("SimilarityScoreMean").fill_null(0.0),
    )
    .with_columns(
        pl.when(pl.col("GrowthNextStageGroup") == "Stay")
        .then(pl.col("GrowthStageGroup"))
        .otherwise(pl.col("GrowthNextStageGroup"))
        .alias("GrowthNextStageGroup")
    )
)
del panel, stat_concorrenti, stat_simili
gc.collect()

mappa_id = finale.select("CompanyID").unique().sort("CompanyID").with_row_index("_nuovo", offset=1)
finale = finale.join(mappa_id, on="CompanyID", how="left").drop("CompanyID").rename({"_nuovo": "CompanyID"})

COLONNE_FINALI = [
    "CompanyID", "Age", "YearFounded",
    "GrowthStageGroup", "GrowthNextStageGroup", "TimeNextStageGroup",
    "N_Deal", "TotalRaised", "Percent_Females",
    "Is_Eco", "Is_Eng", "Is_NS", "Is_Hum", "Is_SS", "Is_Med", "Is_Law", "Is_IT",
    "Institute", "WorkExp_Idx_Mean", "Total_Founders",
    "Is_Debt", "Is_SpinOff", "Is_CrowdFunding", "MeanMedianRoundAmount_cum",
    "Is_Accelerator", "has_Corporate", "has_VentureCapital", "has_PublicInvestor",
    "has_Angel_Lead", "has_Corporate_Lead", "has_VentureCapital_Lead",
    "has_Accelerator_Lead", "has_PrivateEquity_Lead", "has_PublicInvestor_Lead",
    "HQCountry", "PrimaryIndustrySector",
    "SimilarityScoreMean", "N_Competitors", "Same_Country",
    "Highest_Degree_CEO", "Gender_CEO", "MeanTotalInvestments_cum", "WorkExperienceIndex_CEO",
    "Is_Angel", "Total_People", "Is_Grant", "has_PrivateEquity", "TotalInvestors",
    "Highest_Degree_Mean", "Avg_Earliest_Year",
    "UndisclosedAmountShare",
    "N_Similar",
]
finale = finale.select(COLONNE_FINALI)

finale.write_parquet(cfg.interim("panel.parquet"))
finale.write_csv(cfg.interim("panel.csv.gz"), compression="gzip")
print(f"panel finale: {finale.height:,} righe x {finale.width} colonne")
print(f"'Stay' residui: {(finale['GrowthNextStageGroup'] == 'Stay').sum()}   (deve essere 0)")

panel finale: 802,148 righe x 52 colonne
'Stay' residui: 0   (deve essere 0)


---
## Verifica

L'ultimo blocco controlla le **invarianti** del panel: righe, aziende, colonne,
quante righe hanno dati di team e uno stadio di crescita, più due controlli di
merito — nessun anno precedente alla fondazione, e la coorte più vecchia deve
avere dati di team — e che le colonne coincidano con lo schema d'esempio.

I valori attesi appartengono a **questa estrazione**: uno scostamento significa «è
cambiato qualcosa, capisci cosa prima di usare il risultato», non necessariamente
«c'è un errore». La logica, invece, è coperta dai test: `uv run pytest`.

In [47]:
# Verifica delle invarianti
finale = pl.read_parquet(cfg.interim("panel.parquet"))

ATTESI = {
    "righe": 802_148,
    "aziende": 116_312,
    "colonne": 52,
    "righe con dati di team": 700_699,
    "righe con stadio": 561_333,
}
ottenuti = {
    "righe": finale.height,
    "aziende": finale["CompanyID"].n_unique(),
    "colonne": finale.width,
    "righe con dati di team": int(finale["Total_People"].is_not_null().sum()),
    "righe con stadio": int(finale["GrowthStageGroup"].is_not_null().sum()),
}
for nome, atteso in ATTESI.items():
    ott = ottenuti[nome]
    stato = "ok" if ott == atteso else f"ATTESO {atteso:,}"
    print(f"  {nome:<24}{ott:>10,}   {stato}")

assert finale.filter(pl.col("Age") < 0).height == 0, "ci sono anni precedenti alla fondazione"
coorte2000 = finale.filter(pl.col("YearFounded") == 2000)
quota = 100 * coorte2000["Total_People"].is_not_null().sum() / coorte2000.height
assert quota > 70, f"la coorte 2000 ha solo il {quota:.1f}% di righe con team"
print(f"\n  righe con Age < 0            0   ok")
print(f"  coorte 2000 con team      {quota:5.1f}%  ok")

esempio = pl.read_csv("data/raw/example_panel.csv", infer_schema_length=0).columns
# Nello schema d'esempio le colonne competitor compaiono in due versioni; qui
# ne esiste una sola, ed e' l'interruttore a deciderne il contenuto.
atteso_colonne = [c if c != "TotalRaised_Est" else "TotalRaised"
                  for c in esempio if not c.endswith("_All")]
atteso_colonne += ["UndisclosedAmountShare", "N_Similar"]
assert finale.columns == atteso_colonne, "le colonne non coincidono con example_panel"
print("  colonne: quelle dello schema d'esempio, piu' le due aggiunte   ok")

if all(ottenuti[k] == v for k, v in ATTESI.items()):
    print("\nTUTTE LE INVARIANTI RISPETTATE.")
else:
    print("\nQUALCOSA E' CAMBIATO: confronta con docs/panel_revisione_stato.md.")

  righe                      802,148   ok
  aziende                    116,312   ok
  colonne                         52   ok
  righe con dati di team     700,699   ok
  righe con stadio           561,333   ok

  righe con Age < 0            0   ok
  coorte 2000 con team       78.4%  ok


  colonne: quelle dello schema d'esempio, piu' le due aggiunte   ok

TUTTE LE INVARIANTI RISPETTATE.
